# BNPL Executive Risk & Business Translation

### Credit Loss Forecasting and Stress Testing for BNPL Lending

This notebook translates the project's frozen BNPL credit-risk outputs into an executive portfolio-risk and decision framework.

The analysis brings together:

- transaction-level Probability of Default (PD)
- Exposure at Default (EAD)
- risk-band concentration
- customer behavioural segmentation
- early-warning indicators
- scenario-based Expected Loss
- macroeconomic stress testing
- external real-world risk evidence

### Executive objective

The analysis is designed to answer four management questions:

1. **Where is portfolio risk concentrated?**
2. **Which exposures and customer groups require differentiated management?**
3. **How could portfolio losses change under adverse conditions?**
4. **What actions should management consider across approval, monitoring, collections, exposure management and stress preparedness?**

### Analytical flow

**BNPL Model Outputs**  
↓  
**Risk Bands & Customer Segments**  
↓  
**Portfolio Exposure & Risk Concentration**  
↓  
**Expected Loss**  
↓  
**Stress Testing**  
↓  
**Management Decisions**


The BNPL dataset is synthetic. Results are therefore presented as analytical decision-support evidence and are not interpreted as estimates of actual Nigerian BNPL market default rates or losses.

## 1. Executive Interpretation Framework

The analysis distinguishes between three levels of evidence.

### 1.1 Executive KPIs

These quantify the economic position and resilience of the portfolio.

- Total Portfolio EAD
- Exposure-Weighted PD
- Baseline Expected Loss
- EL / EAD
- Very High Risk EAD Share
- Very High PD-Weighted Exposure Share
- Very High Expected Loss Share
- Risk Concentration Ratio
- Moderate Stress Expected Loss
- Severe Stress Expected Loss
- Severe / Baseline Expected Loss Multiple

### 1.2 Business Diagnostics

These explain where and why management attention should be concentrated.

- risk-band exposure
- risk-band default outcomes
- customer-segment exposure
- customer-segment risk concentration
- early-warning indicator lifts
- stress impacts by risk band and segment

### 1.3 Technical Evidence

These establish analytical credibility but are not treated as executive KPIs.

- ROC-AUC
- PR-AUC
- Recall
- Precision
- F1
- KS
- Gini
- model threshold
- feature importance
- clustering silhouette score

The executive narrative prioritises **economic significance and decision relevance** over model metrics alone.

## 2. Executive KPI Dictionary

| KPI | Definition | Management meaning |
|---|---|---|
| **Total Portfolio EAD** | Σ EAD | Total exposure represented in the analysed portfolio |
| **Exposure-Weighted PD** | Σ(PD × EAD) / ΣEAD | Default risk weighted by economic exposure |
| **Baseline Expected Loss** | Σ(PD × EAD × LGD) | Scenario-based expected portfolio loss under baseline LGD |
| **EL / EAD** | Expected Loss / EAD | Expected loss relative to portfolio exposure |
| **Very High Risk EAD Share** | Very High EAD / Total EAD | Exposure concentrated in the highest risk tier |
| **Very High PD-Weighted Exposure Share** | Very High Σ(PD × EAD) / Total Σ(PD × EAD) | Share of risk-weighted exposure concentrated in the Very High tier |
| **Very High Expected Loss Share** | Very High EL / Total EL | Share of baseline expected loss attributable to the Very High tier |
| **Risk Concentration Ratio** | Very High PD-weighted exposure share / Very High EAD share | Degree of disproportionate risk concentration |
| **Moderate Stress Expected Loss** | Stressed Σ(PD × EAD × LGD) | Portfolio loss under moderate adverse assumptions |
| **Severe Stress Expected Loss** | Stressed Σ(PD × EAD × LGD) | Portfolio loss under severe adverse assumptions |
| **Severe / Baseline EL Multiple** | Severe EL / Baseline EL | Magnitude of loss escalation under severe stress |

### Interpretation rules

- **Exposure-Weighted PD** is the headline portfolio PD measure.
- Transaction-weighted PD is retained as a supporting diagnostic rather than a headline KPI.
- Risk bands are empirical, distribution-based risk tiers and are not regulatory grades.
- Observed default rates are reported separately from model-derived PD.
- EAD uses principal-based exposure because reliable outstanding balance at default is unavailable.
- LGD is scenario-based because reliable recovery data is unavailable.
- Expected Loss is a simplified scenario-based portfolio measure, not an IFRS-compliant accounting ECL estimate.
- Macro stress scenarios are used for sensitivity and resilience analysis, not causal macroeconomic default forecasting.

> **Decision principle:** A strong executive risk indicator should connect portfolio exposure, risk intensity and potential loss, rather than relying on model scores alone.

In [0]:
# ============================================================================
# FROZEN ANALYTICAL INPUTS
# ============================================================================
# Purpose:
# Load only frozen outputs from the preceding analytical notebooks.
#
# This notebook is an executive/business translation layer.
# It does NOT retrain models, redefine risk bands, modify segmentation,
# or recalculate the underlying portfolio risk methodology.
# ============================================================================

from pyspark.sql import functions as F


# ----------------------------------------------------------------------------
# Project Output Paths
# ----------------------------------------------------------------------------

BNPL_RAW_PATH = "/Volumes/workspace/default/bnpl_raw"


# Portfolio-level BNPL risk output from Notebook 08
PORTFOLIO_RISK_PATH = (
    f"{BNPL_RAW_PATH}/bnpl_portfolio_risk"
)


# Authoritative five-band portfolio risk summary from Notebook 08
RISK_BAND_PATH = (
    f"{BNPL_RAW_PATH}/bnpl_portfolio_risk_band_summary"
)


# Final customer segmentation output from Notebook 04
SEGMENT_PATH = (
    f"{BNPL_RAW_PATH}/final_bnpl_customer_segments"
)


# Baseline / scenario expected-loss portfolio summary from Notebook 08
EXPECTED_LOSS_PATH = (
    f"{BNPL_RAW_PATH}/bnpl_expected_loss_portfolio_summary"
)


# Portfolio-level macro stress summary from Notebook 08
STRESS_PATH = (
    f"{BNPL_RAW_PATH}/bnpl_macro_stress_portfolio_summary"
)


# Risk-band stress results from Notebook 08
STRESS_RISK_BAND_PATH = (
    f"{BNPL_RAW_PATH}/bnpl_macro_stress_risk_band"
)


# Customer-segment stress results from Notebook 08
STRESS_SEGMENT_PATH = (
    f"{BNPL_RAW_PATH}/bnpl_macro_stress_segment"
)


# ----------------------------------------------------------------------------
# Load Frozen Delta Outputs
# ----------------------------------------------------------------------------

portfolio_risk = (
    spark.read
    .format("delta")
    .load(PORTFOLIO_RISK_PATH)
)


risk_band_concentration = (
    spark.read
    .format("delta")
    .load(RISK_BAND_PATH)
)


segment_concentration = (
    spark.read
    .format("delta")
    .load(SEGMENT_PATH)
)


expected_loss = (
    spark.read
    .format("delta")
    .load(EXPECTED_LOSS_PATH)
)


stress_summary = (
    spark.read
    .format("delta")
    .load(STRESS_PATH)
)


stress_risk_band = (
    spark.read
    .format("delta")
    .load(STRESS_RISK_BAND_PATH)
)


stress_segment = (
    spark.read
    .format("delta")
    .load(STRESS_SEGMENT_PATH)
)


# ----------------------------------------------------------------------------
# Frozen Input Inventory
# ----------------------------------------------------------------------------

frozen_inputs = [
    ("Portfolio Risk", PORTFOLIO_RISK_PATH),
    ("Risk-Band Summary", RISK_BAND_PATH),
    ("Customer Segments", SEGMENT_PATH),
    ("Expected Loss Summary", EXPECTED_LOSS_PATH),
    ("Macro Stress Summary", STRESS_PATH),
    ("Risk-Band Stress", STRESS_RISK_BAND_PATH),
    ("Segment Stress", STRESS_SEGMENT_PATH),
]


print("Frozen analytical inputs loaded successfully.")
print()
print("Input inventory:")
for input_name, input_path in frozen_inputs:
    print(f"  ✓ {input_name}: {input_path}")

Frozen analytical inputs loaded successfully.

Input inventory:
  ✓ Portfolio Risk: /Volumes/workspace/default/bnpl_raw/bnpl_portfolio_risk
  ✓ Risk-Band Summary: /Volumes/workspace/default/bnpl_raw/bnpl_portfolio_risk_band_summary
  ✓ Customer Segments: /Volumes/workspace/default/bnpl_raw/final_bnpl_customer_segments
  ✓ Expected Loss Summary: /Volumes/workspace/default/bnpl_raw/bnpl_expected_loss_portfolio_summary
  ✓ Macro Stress Summary: /Volumes/workspace/default/bnpl_raw/bnpl_macro_stress_portfolio_summary
  ✓ Risk-Band Stress: /Volumes/workspace/default/bnpl_raw/bnpl_macro_stress_risk_band
  ✓ Segment Stress: /Volumes/workspace/default/bnpl_raw/bnpl_macro_stress_segment


## 3.1 Input Governance

All downstream analysis in this notebook uses frozen outputs produced by the preceding analytical stages.

The notebook does not:

- retrain the BNPL model
- change the selected risk threshold
- redefine the five risk bands
- modify customer segmentation
- alter Expected Loss assumptions
- introduce Home Credit customer-level integration

This preserves analytical lineage from modelling through executive decision-making.

# 4. Executive Portfolio Scorecard

The portfolio scorecard provides a consolidated view of the BNPL portfolio's economic risk position.

The scorecard combines:

- portfolio exposure
- exposure-weighted default risk
- expected loss
- risk concentration
- stress-loss sensitivity

These measures are intended to provide management with a concise view of **risk magnitude, risk concentration and potential loss impact**.

### Executive reading sequence

**Exposure → Risk → Loss → Concentration → Stress**

This sequence ensures that portfolio risk is interpreted in economic terms rather than through predictive-model metrics alone.

In [0]:
# ============================================================================
# 4.1 EXECUTIVE PORTFOLIO KPI SCORECARD
# ============================================================================

from pyspark.sql import functions as F


# ----------------------------------------------------------------------------
# 4.1.1 Calculate Portfolio-Level Metrics
# ----------------------------------------------------------------------------

portfolio_summary = (
    portfolio_risk
    .agg(
        F.count("*").alias("transaction_count"),
        F.countDistinct("customer_id").alias("customer_count"),
        F.sum("ead_ngn").alias("total_ead_ngn"),
        F.avg("portfolio_pd").alias("transaction_weighted_pd"),
        F.sum(
            F.col("portfolio_pd") * F.col("ead_ngn")
        ).alias("pd_weighted_exposure_ngn")
    )
    .collect()[0]
)


# ----------------------------------------------------------------------------
# 4.1.2 Extract and Explicitly Cast Metrics
# ----------------------------------------------------------------------------
# Explicit float conversion prevents Spark from trying to merge LongType
# and DoubleType when constructing the executive KPI DataFrame.
# ----------------------------------------------------------------------------

transaction_count = int(portfolio_summary["transaction_count"])
customer_count = int(portfolio_summary["customer_count"])

total_ead_ngn = float(portfolio_summary["total_ead_ngn"])
transaction_weighted_pd = float(
    portfolio_summary["transaction_weighted_pd"]
)
pd_weighted_exposure_ngn = float(
    portfolio_summary["pd_weighted_exposure_ngn"]
)


# ----------------------------------------------------------------------------
# 4.1.3 Calculate Exposure-Weighted PD
# ----------------------------------------------------------------------------

exposure_weighted_pd = (
    pd_weighted_exposure_ngn / total_ead_ngn
    if total_ead_ngn > 0
    else 0.0
)


# ----------------------------------------------------------------------------
# 4.1.4 Build Executive KPI Table
# ----------------------------------------------------------------------------

executive_kpis = [
    (
        "Total Portfolio EAD",
        float(total_ead_ngn),
        "NGN",
        "Total exposure represented by the analysed BNPL portfolio"
    ),
    (
        "Exposure-Weighted PD",
        float(exposure_weighted_pd),
        "Percentage",
        "Economic portfolio-level probability of default weighted by exposure"
    ),
    (
        "Transaction-Weighted PD",
        float(transaction_weighted_pd),
        "Percentage",
        "Supporting diagnostic showing average transaction-level PD"
    ),
    (
        "PD-Weighted Exposure",
        float(pd_weighted_exposure_ngn),
        "NGN",
        "Exposure-weighted amount at risk before applying LGD"
    ),
    (
        "Transaction Count",
        float(transaction_count),
        "Count",
        "Number of BNPL transactions in the frozen portfolio"
    ),
    (
        "Customer Count",
        float(customer_count),
        "Count",
        "Number of distinct customers represented in the portfolio"
    )
]


# ----------------------------------------------------------------------------
# 4.1.5 Create Spark DataFrame with Explicit Schema
# ----------------------------------------------------------------------------

executive_kpi_df = spark.createDataFrame(
    executive_kpis,
    schema="""
        kpi STRING,
        value DOUBLE,
        unit STRING,
        executive_interpretation STRING
    """
)


# ----------------------------------------------------------------------------
# 4.1.6 Create Executive Display
# ----------------------------------------------------------------------------

executive_kpi_display = (
    executive_kpi_df
    .withColumn(
        "value_display",
        F.when(
            F.col("unit") == "NGN",
            F.concat(
                F.lit("₦"),
                F.format_number(F.col("value"), 0)
            )
        )
        .when(
            F.col("unit") == "Percentage",
            F.concat(
                F.format_number(F.col("value") * 100, 2),
                F.lit("%")
            )
        )
        .when(
            F.col("unit") == "Count",
            F.format_number(F.col("value"), 0)
        )
        .otherwise(
            F.col("value").cast("string")
        )
    )
    .select(
        "kpi",
        "value_display",
        "executive_interpretation"
    )
)


display(executive_kpi_display)


# ----------------------------------------------------------------------------
# 4.1.7 Retain Frozen Metrics for Subsequent Executive Sections
# ----------------------------------------------------------------------------

print("\nExecutive portfolio scorecard calculated successfully.")
print(f"Total Portfolio EAD  : ₦{total_ead_ngn:,.0f}")
print(f"Exposure-Weighted PD : {exposure_weighted_pd:.2%}")
print(f"PD-Weighted Exposure : ₦{pd_weighted_exposure_ngn:,.0f}")
print(f"Transactions         : {transaction_count:,}")
print(f"Customers            : {customer_count:,}")

kpi,value_display,executive_interpretation
Total Portfolio EAD,"₦33,317,913,043",Total exposure represented by the analysed BNPL portfolio
Exposure-Weighted PD,9.87%,Economic portfolio-level probability of default weighted by exposure
Transaction-Weighted PD,8.00%,Supporting diagnostic showing average transaction-level PD
PD-Weighted Exposure,"₦3,289,674,923",Exposure-weighted amount at risk before applying LGD
Transaction Count,"666,246",Number of BNPL transactions in the frozen portfolio
Customer Count,"421,457",Number of distinct customers represented in the portfolio



Executive portfolio scorecard calculated successfully.
Total Portfolio EAD  : ₦33,317,913,043
Exposure-Weighted PD : 9.87%
PD-Weighted Exposure : ₦3,289,674,923
Transactions         : 666,246
Customers            : 421,457


## 4.2 Baseline Expected Loss

The baseline expected-loss view translates portfolio probability of default (PD),
exposure at default (EAD), and scenario-based loss given default (LGD) into a
single portfolio loss estimate.

Because the BNPL dataset does not contain reliable recovery information, LGD is
treated as a transparent scenario assumption rather than an empirically estimated
recovery rate.

Baseline Expected Loss = Σ(PDᵢ × EADᵢ × LGD)

This is a simplified scenario-based portfolio expected-loss estimate and should
not be interpreted as IFRS-grade accounting expected credit loss.

In [0]:
# ============================================================
# 4.2 BASELINE EXPECTED LOSS
# ============================================================

from pyspark.sql import functions as F

# Notebook 08 already produced the authoritative portfolio
# expected-loss summary. We consume the frozen output here.

baseline_expected_loss = (
    expected_loss
    .filter(F.col("lgd_scenario") == "Baseline")
    .select(
        "lgd",
        "pd_ead_exposure_ngn",
        "expected_loss_ngn",
        "expected_loss_rate_of_ead"
    )
)

display(baseline_expected_loss)

lgd,pd_ead_exposure_ngn,expected_loss_ngn,expected_loss_rate_of_ead
0.4,3.289674922513504E9,1.3158699690054088E9,0.03949436950986159


## 4.3 Executive Expected Loss Metrics

The baseline expected-loss output is converted into executive-level economic
metrics for use throughout the business translation layer.

The objective is to distinguish:

- total portfolio exposure,
- PD-weighted exposure,
- baseline expected loss,
- expected loss as a percentage of EAD,
- and the assumed baseline LGD.

These metrics provide the economic bridge between modelled risk and portfolio
decision-making.

In [0]:
# ============================================================
# 4.3 EXECUTIVE EXPECTED LOSS METRICS
# ============================================================

# Collect the single baseline row once.
baseline_row = baseline_expected_loss.first()

if baseline_row is None:
    raise ValueError(
        "No Baseline expected-loss record found in Notebook 08 output."
    )

# Extract frozen baseline metrics
baseline_lgd = float(baseline_row["lgd"])
baseline_pd_ead_exposure_ngn = float(
    baseline_row["pd_ead_exposure_ngn"]
)
baseline_expected_loss_ngn = float(
    baseline_row["expected_loss_ngn"]
)
baseline_el_rate_of_ead = float(
    baseline_row["expected_loss_rate_of_ead"]
)

# Total portfolio EAD comes from the authoritative portfolio-risk output.
portfolio_ead_row = (
    portfolio_risk
    .agg(F.sum("ead_ngn").alias("total_ead_ngn"))
    .first()
)

if portfolio_ead_row is None:
    raise ValueError(
        "Unable to calculate total portfolio EAD from portfolio_risk."
    )

baseline_ead_ngn = float(portfolio_ead_row["total_ead_ngn"])

# Executive metrics table
expected_loss_metrics = [
    (
        "Total Portfolio EAD",
        baseline_ead_ngn,
        "NGN",
        "Total principal-based exposure represented in the portfolio."
    ),
    (
        "PD-Weighted Exposure",
        baseline_pd_ead_exposure_ngn,
        "NGN",
        "Exposure-weighted portfolio risk before applying LGD."
    ),
    (
        "Baseline Expected Loss",
        baseline_expected_loss_ngn,
        "NGN",
        "Scenario-based expected loss using baseline LGD."
    ),
    (
        "Expected Loss / EAD",
        baseline_el_rate_of_ead,
        "%",
        "Baseline expected loss relative to total portfolio exposure."
    ),
    (
        "Baseline LGD",
        baseline_lgd,
        "%",
        "Scenario assumption for loss severity under the baseline case."
    )
]

expected_loss_metrics_df = spark.createDataFrame(
    expected_loss_metrics,
    schema=[
        "metric",
        "value",
        "unit",
        "executive_interpretation"
    ]
)

display(
    expected_loss_metrics_df
    .withColumn(
        "display_value",
        F.when(
            F.col("unit") == "NGN",
            F.concat(
                F.lit("₦"),
                F.format_number(F.col("value"), 0)
            )
        )
        .when(
            F.col("unit") == "%",
            F.concat(
                F.format_number(F.col("value") * 100, 2),
                F.lit("%")
            )
        )
        .otherwise(
            F.format_number(F.col("value"), 2)
        )
    )
    .select(
        "metric",
        "display_value",
        "executive_interpretation"
    )
)

metric,display_value,executive_interpretation
Total Portfolio EAD,"₦33,317,913,043",Total principal-based exposure represented in the portfolio.
PD-Weighted Exposure,"₦3,289,674,923",Exposure-weighted portfolio risk before applying LGD.
Baseline Expected Loss,"₦1,315,869,969",Scenario-based expected loss using baseline LGD.
Expected Loss / EAD,3.95%,Baseline expected loss relative to total portfolio exposure.
Baseline LGD,40.00%,Scenario assumption for loss severity under the baseline case.


## 4.4 Macro Stress Summary

The stress-testing layer translates adverse macroeconomic assumptions into
portfolio-level expected-loss sensitivity.

The analysis uses three scenarios:

- Baseline
- Moderate stress
- Severe stress

The outputs are consumed from the frozen Portfolio Risk Engine. No new
macro-default model is estimated at this stage.

These results represent scenario sensitivity, not a causal forecast of how
macroeconomic variables will affect BNPL defaults.

In [0]:
# ============================================================
# 4.4 MACRO STRESS SUMMARY
# ============================================================

from pyspark.sql import functions as F

# Select the authoritative portfolio-level stress outputs
stress_summary_exec = (
    stress_summary
    .select(
        "scenario",
        "stressed_ead_ngn",
        "stressed_pd_ead_exposure_ngn",
        "stressed_expected_loss_ngn",
        "incremental_expected_loss_ngn",
        "stress_expected_loss_multiple",
        "stress_expected_loss_increase_pct"
    )
    .orderBy(
        F.when(F.col("scenario") == "Baseline", 1)
         .when(F.col("scenario") == "Moderate", 2)
         .when(F.col("scenario") == "Severe", 3)
         .otherwise(99)
    )
)

display(stress_summary_exec)

scenario,stressed_ead_ngn,stressed_pd_ead_exposure_ngn,stressed_expected_loss_ngn,incremental_expected_loss_ngn,stress_expected_loss_multiple,stress_expected_loss_increase_pct
Baseline,3.3317913042690228E10,3.28967492251355E9,1.315869969005415E9,1.5020370483398438E-5,1.0000000000000113,1.1324274851176597E-12
Severe Macro Stress,3.3317913042690308E10,4.93451238377032E9,3.4541586686392164E9,2.1382886996338165E9,2.6250000000000315,162.50000000000315
Moderate Macro Stress,3.3317913042690308E10,4.112093653141924E9,2.2616515092280593E9,9.457815402226593E8,1.7187500000000213,71.87500000000213


In [0]:
# ============================================================
# EXECUTIVE STRESS SCORECARD
# ============================================================

stress_scorecard = (
    stress_summary_exec
    .select(
        "scenario",
        F.format_number(
            F.col("stressed_ead_ngn"), 0
        ).alias("EAD_NGN"),
        F.format_number(
            F.col("stressed_pd_ead_exposure_ngn"), 0
        ).alias("PD_WEIGHTED_EXPOSURE_NGN"),
        F.format_number(
            F.col("stressed_expected_loss_ngn"), 0
        ).alias("EXPECTED_LOSS_NGN"),
        F.format_number(
            F.col("incremental_expected_loss_ngn"), 0
        ).alias("INCREMENTAL_EL_NGN"),
        F.format_number(
            F.col("stress_expected_loss_multiple"), 2
        ).alias("EL_MULTIPLE"),
        F.format_number(
            F.col("stress_expected_loss_increase_pct"), 2
        ).alias("EL_INCREASE_PCT")
    )
)

display(stress_scorecard)

scenario,EAD_NGN,PD_WEIGHTED_EXPOSURE_NGN,EXPECTED_LOSS_NGN,INCREMENTAL_EL_NGN,EL_MULTIPLE,EL_INCREASE_PCT
Baseline,"33,317,913,043","3,289,674,923","1,315,869,969",0,1.00,0.00
Severe Macro Stress,"33,317,913,043","4,934,512,384","3,454,158,669","2,138,288,700",2.63,162.50
Moderate Macro Stress,"33,317,913,043","4,112,093,653","2,261,651,509","945,781,540",1.72,71.88


## 4.5 Portfolio Risk Concentration

Portfolio-level averages can conceal concentration of risk.

This section evaluates how exposure and PD-weighted exposure are distributed
across the five empirical risk bands.

Particular attention is given to the Very High Risk band because concentration
of PD-weighted exposure and expected loss in this band has direct implications
for underwriting, monitoring, collections and exposure management.

Risk bands are distribution-based portfolio tiers derived from the modelled PD
distribution. They are not regulatory risk grades or fixed PD cut-offs.

In [0]:
# ============================================================
# 4.5 PORTFOLIO RISK CONCENTRATION
# ============================================================

from pyspark.sql import functions as F

# Authoritative risk-band summary from Notebook 08
risk_concentration = (
    risk_band_concentration
    .select(
        "risk_band",
        "risk_band_order",
        "transaction_count",
        "customer_count",
        "ead_ngn",
        "average_pd",
        "exposure_weighted_pd",
        "pd_weighted_exposure_ngn",
        "observed_default_count",
        "observed_default_exposure_ngn",
        "ead_share_pct",
        "pd_weighted_exposure_share_pct"
    )
    .orderBy("risk_band_order")
)

display(risk_concentration)

risk_band,risk_band_order,transaction_count,customer_count,ead_ngn,average_pd,exposure_weighted_pd,pd_weighted_exposure_ngn,observed_default_count,observed_default_exposure_ngn,ead_share_pct,pd_weighted_exposure_share_pct
Very Low,1,133214,121278,4.996942780677462E9,0.023637779717307888,0.023640075981890777,1.1812810701237577E8,0.0,0.0,14.9977664395572,3.5908747762261566
Low,2,133296,120782,5.310990428719698E9,0.03471043708653736,0.03501492552399123,1.8596393432025048E8,0.0,0.0,15.940345428943017,5.6529577754801865
Moderate,3,133206,119963,8.160189383352885E9,0.051723074288344176,0.052185431768723554,4.258430062848243E8,35.0,1322145.1858605405,24.4918983157715,12.94483547205484
High,4,133290,120632,6.462766710921511E9,0.07877612773939378,0.0798119168996802,5.1580579967408717E8,1476.0,7.267217819862382E7,19.397273480607115,15.679537091767628
Very High,5,133240,120673,8.387023739018707E9,0.21123131560202968,0.2437019542120844,2.043934075222002E9,51799.0,3.7850521176387434E9,25.172716335120963,62.13179488447144


In [0]:
# ============================================================
# VERY HIGH RISK CONCENTRATION KPIs
# ============================================================

very_high_row = (
    risk_concentration
    .filter(F.col("risk_band") == "Very High")
    .first()
)

if very_high_row is None:
    raise ValueError(
        "Very High risk band was not found in the authoritative "
        "risk-band summary."
    )

very_high_ead_share = float(
    very_high_row["ead_share_pct"]
) / 100

very_high_pd_weighted_share = float(
    very_high_row["pd_weighted_exposure_share_pct"]
) / 100

very_high_pd_weighted_exposure_ngn = float(
    very_high_row["pd_weighted_exposure_ngn"]
)

# Because baseline LGD is a portfolio-level scenario assumption,
# baseline EL contribution is proportional to PD-weighted exposure.
very_high_expected_loss_ngn = (
    very_high_pd_weighted_exposure_ngn * baseline_lgd
)

very_high_expected_loss_share = (
    very_high_expected_loss_ngn / baseline_expected_loss_ngn
)

risk_concentration_ratio = (
    very_high_pd_weighted_share / very_high_ead_share
)

print("Very High Risk EAD Share:",
      f"{very_high_ead_share:.2%}")

print("Very High PD-Weighted Exposure Share:",
      f"{very_high_pd_weighted_share:.2%}")

print("Very High Expected Loss Share:",
      f"{very_high_expected_loss_share:.2%}")

print("Risk Concentration Ratio:",
      f"{risk_concentration_ratio:.2f}x")

Very High Risk EAD Share: 25.17%
Very High PD-Weighted Exposure Share: 62.13%
Very High Expected Loss Share: 62.13%
Risk Concentration Ratio: 2.47x


In [0]:
# ============================================================
# EXECUTIVE RISK CONCENTRATION SCORECARD
# ============================================================

risk_concentration_kpis = [
    (
        "Very High Risk EAD Share",
        very_high_ead_share,
        "%",
        "Share of total portfolio exposure concentrated in the Very High band."
    ),
    (
        "Very High PD-Weighted Exposure Share",
        very_high_pd_weighted_share,
        "%",
        "Share of portfolio PD-weighted exposure concentrated in the Very High band."
    ),
    (
        "Very High Expected Loss Share",
        very_high_expected_loss_share,
        "%",
        "Share of baseline expected loss attributable to the Very High band."
    ),
    (
        "Risk Concentration Ratio",
        risk_concentration_ratio,
        "x",
        "Relative concentration of PD-weighted exposure versus EAD in the Very High band."
    )
]

risk_concentration_kpi_df = spark.createDataFrame(
    risk_concentration_kpis,
    schema=[
        "kpi",
        "value",
        "unit",
        "executive_interpretation"
    ]
)

display(risk_concentration_kpi_df)

kpi,value,unit,executive_interpretation
Very High Risk EAD Share,0.25172716335120965,%,Share of total portfolio exposure concentrated in the Very High band.
Very High PD-Weighted Exposure Share,0.6213179488447144,%,Share of portfolio PD-weighted exposure concentrated in the Very High band.
Very High Expected Loss Share,0.6213179488447163,%,Share of baseline expected loss attributable to the Very High band.
Risk Concentration Ratio,2.4682197208008567,x,Relative concentration of PD-weighted exposure versus EAD in the Very High band.


## 4.6 Risk × Exposure Decision Matrix

The risk-band results are translated into portfolio-management actions.

The framework combines modelled risk and exposure concentration to distinguish
between growth opportunities, enhanced monitoring, exposure controls and
collections priorities.

These recommendations are decision rules derived from the portfolio analysis.
They are not automated underwriting policies and should be validated against
business policy, affordability requirements and operational constraints before
implementation.

In [0]:
# ============================================================
# 4.6 RISK × EXPOSURE DECISION MATRIX
# ============================================================

from pyspark.sql import functions as F

# Start from the authoritative risk-band summary produced by Notebook 08.
decision_matrix = (
    risk_concentration
    .select(
        "risk_band",
        "risk_band_order",
        "transaction_count",
        "customer_count",
        "ead_ngn",
        "ead_share_pct",
        "average_pd",
        "exposure_weighted_pd",
        "pd_weighted_exposure_ngn",
        "pd_weighted_exposure_share_pct",
        "observed_default_count",
        "observed_default_exposure_ngn"
    )
    .orderBy("risk_band_order")
)

# ------------------------------------------------------------
# Management action framework
# ------------------------------------------------------------

decision_rules = {
    "Very Low": (
        "Controlled growth",
        "Standard monitoring",
        "Maintain standard limits; selectively support repeat borrowing"
    ),
    "Low": (
        "Risk-aware growth",
        "Standard monitoring",
        "Maintain normal exposure with routine performance monitoring"
    ),
    "Moderate": (
        "Selective growth",
        "Enhanced monitoring",
        "Review limits and monitor repayment behaviour more closely"
    ),
    "High": (
        "Exposure control",
        "High-frequency monitoring",
        "Tighten incremental exposure and strengthen early-warning monitoring"
    ),
    "Very High": (
        "Protect capital",
        "Priority collections",
        "Restrict incremental exposure and prioritise collections/intervention"
    )
}

# Create decision rows in the same risk-band order.
decision_rows = []

for row in decision_matrix.collect():
    band = row["risk_band"]

    if band not in decision_rules:
        raise ValueError(
            f"No management rule defined for risk band: {band}"
        )

    growth_action, monitoring_action, exposure_action = decision_rules[band]

    decision_rows.append(
        (
            band,
            int(row["risk_band_order"]),
            float(row["ead_share_pct"]),
            float(row["average_pd"]),
            float(row["exposure_weighted_pd"]),
            float(row["pd_weighted_exposure_share_pct"]),
            growth_action,
            monitoring_action,
            exposure_action
        )
    )

decision_matrix_df = spark.createDataFrame(
    decision_rows,
    schema=[
        "risk_band",
        "risk_band_order",
        "ead_share_pct",
        "average_pd",
        "exposure_weighted_pd",
        "pd_weighted_exposure_share_pct",
        "growth_action",
        "monitoring_action",
        "exposure_action"
    ]
)

display(
    decision_matrix_df
    .orderBy("risk_band_order")
)

risk_band,risk_band_order,ead_share_pct,average_pd,exposure_weighted_pd,pd_weighted_exposure_share_pct,growth_action,monitoring_action,exposure_action
Very Low,1,14.9977664395572,0.023637779717307888,0.023640075981890777,3.5908747762261566,Controlled growth,Standard monitoring,Maintain standard limits; selectively support repeat borrowing
Low,2,15.940345428943017,0.03471043708653736,0.03501492552399123,5.6529577754801865,Risk-aware growth,Standard monitoring,Maintain normal exposure with routine performance monitoring
Moderate,3,24.4918983157715,0.051723074288344176,0.052185431768723554,12.94483547205484,Selective growth,Enhanced monitoring,Review limits and monitor repayment behaviour more closely
High,4,19.397273480607115,0.07877612773939378,0.0798119168996802,15.679537091767628,Exposure control,High-frequency monitoring,Tighten incremental exposure and strengthen early-warning monitoring
Very High,5,25.172716335120963,0.21123131560202968,0.2437019542120844,62.13179488447144,Protect capital,Priority collections,Restrict incremental exposure and prioritise collections/intervention


In [0]:
# ============================================================
# EXECUTIVE DECISION VIEW
# ============================================================

executive_decision_matrix = (
    decision_matrix_df
    .withColumn(
        "EAD Share",
        F.concat(
            F.format_number(F.col("ead_share_pct"), 2),
            F.lit("%")
        )
    )
    .withColumn(
        "Average PD",
        F.concat(
            F.format_number(F.col("average_pd") * 100, 2),
            F.lit("%")
        )
    )
    .select(
        "risk_band",
        "EAD Share",
        "Average PD",
        "growth_action",
        "monitoring_action",
        "exposure_action"
    )
)

display(executive_decision_matrix)

risk_band,EAD Share,Average PD,growth_action,monitoring_action,exposure_action
Very Low,15.00%,2.36%,Controlled growth,Standard monitoring,Maintain standard limits; selectively support repeat borrowing
Low,15.94%,3.47%,Risk-aware growth,Standard monitoring,Maintain normal exposure with routine performance monitoring
Moderate,24.49%,5.17%,Selective growth,Enhanced monitoring,Review limits and monitor repayment behaviour more closely
High,19.40%,7.88%,Exposure control,High-frequency monitoring,Tighten incremental exposure and strengthen early-warning monitoring
Very High,25.17%,21.12%,Protect capital,Priority collections,Restrict incremental exposure and prioritise collections/intervention


### Portfolio Management Priority

The Very High risk band receives the highest management priority because it
combines elevated modelled PD with disproportionate concentration of
PD-weighted exposure.

The objective is not to eliminate lending to this population automatically,
but to shift the management response from growth toward exposure control,
monitoring and collections.

In [0]:
# ============================================================
# MANAGEMENT PRIORITY SUMMARY
# ============================================================

priority_summary = (
    decision_matrix_df
    .withColumn(
        "management_priority",
        F.when(
            F.col("risk_band") == "Very High",
            "CRITICAL"
        )
        .when(
            F.col("risk_band") == "High",
            "HIGH"
        )
        .when(
            F.col("risk_band") == "Moderate",
            "MEDIUM"
        )
        .otherwise("STANDARD")
    )
    .withColumn(
        "portfolio_attention",
        F.when(
            F.col("risk_band") == "Very High",
            "Immediate exposure control and collections focus"
        )
        .when(
            F.col("risk_band") == "High",
            "Enhanced monitoring and tighter incremental exposure"
        )
        .when(
            F.col("risk_band") == "Moderate",
            "Selective growth with enhanced monitoring"
        )
        .otherwise(
            "Routine monitoring with controlled growth"
        )
    )
    .select(
        "risk_band",
        "management_priority",
        "portfolio_attention"
    )
    .orderBy("risk_band_order")
)

display(priority_summary)

risk_band,management_priority,portfolio_attention
Very Low,STANDARD,Routine monitoring with controlled growth
Low,STANDARD,Routine monitoring with controlled growth
Moderate,MEDIUM,Selective growth with enhanced monitoring
High,HIGH,Enhanced monitoring and tighter incremental exposure
Very High,CRITICAL,Immediate exposure control and collections focus


## 4.7 Customer Behaviour & Segmentation

Customer segmentation provides a complementary behavioural view of portfolio risk.

The segmentation was developed independently of the supervised default model using
target-independent customer behavioural characteristics. The purpose is not to
assign customers directly to credit-risk grades, but to identify distinct patterns
of borrowing activity, engagement and exposure that can support differentiated
management strategies.

The final customer segments are profiled using their behavioural characteristics
and then evaluated against portfolio exposure and observed risk outcomes.

### Executive interpretation framework

**Finding → Behavioural implication → Portfolio implication → Management action**

The segments should therefore be interpreted as behavioural profiles rather than
as inherently "good" or "bad" credit groups. Risk differences are evaluated only
after the clusters are formed.

This analysis supports:
- differentiated customer engagement strategies
- exposure and limit management
- monitoring intensity
- targeted early-warning interventions
- risk-adjusted portfolio growth

The segmentation is a secondary customer-level analytical view. The primary BNPL
default model remains transaction-level because the default outcome is attached to
individual transactions.

In [0]:
# ============================================================
# 4.7 Customer Behaviour & Segmentation
# ============================================================

from pyspark.sql import functions as F


# ------------------------------------------------------------
# 4.7.1 Verify frozen segmentation output
# ------------------------------------------------------------

segment_summary = segment_concentration

print("Customer segmentation output:")
segment_summary.printSchema()

print(
    f"Number of behavioural segments: "
    f"{segment_summary.count():,}"
)


# ------------------------------------------------------------
# 4.7.2 Validate required fields
# ------------------------------------------------------------

required_cols = [
    "cluster",
    "customers",
    "transactions",
    "total_exposure_ngn",
    "default_30d_count",
    "default_90d_count",
    "default_30d_rate_pct",
    "default_90d_rate_pct",
    "default_30d_contribution_pct",
    "default_90d_contribution_pct",
    "segment_label"
]

missing_cols = [
    c for c in required_cols
    if c not in segment_summary.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required segmentation columns: {missing_cols}"
    )

print("Required segmentation fields verified.")


# ------------------------------------------------------------
# 4.7.3 Calculate exposure share
# ------------------------------------------------------------

total_segment_exposure = (
    segment_summary
    .agg(
        F.sum("total_exposure_ngn").alias("total_exposure")
    )
    .collect()[0]["total_exposure"]
)

segment_executive = (
    segment_summary
    .withColumn(
        "exposure_share_pct",
        F.col("total_exposure_ngn")
        / F.lit(float(total_segment_exposure))
        * 100
    )
)


# ------------------------------------------------------------
# 4.7.4 Executive behavioural segment profile
# ------------------------------------------------------------

segment_profile = (
    segment_executive
    .select(
        "cluster",
        "segment_label",
        "customers",
        "transactions",
        F.round(
            "total_exposure_ngn",
            2
        ).alias("exposure_ngn"),
        F.round(
            "exposure_share_pct",
            2
        ).alias("exposure_share_pct"),
        F.round(
            "default_30d_rate_pct",
            2
        ).alias("observed_default_30d_pct"),
        F.round(
            "default_90d_rate_pct",
            2
        ).alias("observed_default_90d_pct"),
        F.round(
            "default_90d_contribution_pct",
            2
        ).alias("default_90d_contribution_pct")
    )
    .orderBy("cluster")
)

display(segment_profile)


# ------------------------------------------------------------
# 4.7.5 Management interpretation
# ------------------------------------------------------------

segment_management = (
    segment_executive
    .withColumn(
        "management_priority",
        F.when(
            F.col("segment_label")
            == "Active / Higher-Exposure",
            F.lit("High portfolio priority")
        )
        .when(
            F.col("segment_label")
            == "Low-Engagement / Low-Exposure",
            F.lit("Targeted monitoring")
        )
        .otherwise(
            F.lit("Review segment profile")
        )
    )
    .withColumn(
        "recommended_action",
        F.when(
            F.col("segment_label")
            == "Active / Higher-Exposure",
            F.lit(
                "Prioritise exposure monitoring, repeat-borrowing "
                "controls and early-warning surveillance because "
                "this segment contains most portfolio exposure."
            )
        )
        .when(
            F.col("segment_label")
            == "Low-Engagement / Low-Exposure",
            F.lit(
                "Maintain targeted monitoring while avoiding "
                "unnecessary credit restriction on a relatively "
                "small exposure segment."
            )
        )
        .otherwise(
            F.lit(
                "Review behavioural characteristics before "
                "assigning differentiated management action."
            )
        )
    )
    .select(
        "cluster",
        "segment_label",
        "customers",
        F.round(
            "total_exposure_ngn",
            2
        ).alias("exposure_ngn"),
        F.round(
            "exposure_share_pct",
            2
        ).alias("exposure_share_pct"),
        F.round(
            "default_90d_rate_pct",
            2
        ).alias("observed_default_90d_pct"),
        F.round(
            "default_90d_contribution_pct",
            2
        ).alias("default_90d_contribution_pct"),
        "management_priority",
        "recommended_action"
    )
    .orderBy("cluster")
)

display(segment_management)


# ------------------------------------------------------------
# 4.7.6 Executive takeaway
# ------------------------------------------------------------

active_segment = (
    segment_executive
    .filter(
        F.col("segment_label")
        == "Active / Higher-Exposure"
    )
    .select(
        "exposure_share_pct",
        "default_90d_contribution_pct",
        "default_90d_rate_pct"
    )
    .collect()
)

if active_segment:
    active = active_segment[0]

    print("\n" + "=" * 70)
    print("CUSTOMER SEGMENTATION EXECUTIVE TAKEAWAY")
    print("=" * 70)

    print(
        f"Active / Higher-Exposure represents "
        f"{active['exposure_share_pct']:.2f}% of segment exposure."
    )

    print(
        f"It contributes "
        f"{active['default_90d_contribution_pct']:.2f}% "
        f"of observed 90-day defaults."
    )

    print(
        f"Observed 90-day default rate: "
        f"{active['default_90d_rate_pct']:.2f}%."
    )

    print(
        "\nManagement implication: "
        "prioritise exposure monitoring, repeat-borrowing controls "
        "and early-warning surveillance within the "
        "Active / Higher-Exposure segment."
    )

Customer segmentation output:
root
 |-- cluster: integer (nullable = true)
 |-- customers: long (nullable = true)
 |-- transactions: long (nullable = true)
 |-- total_exposure_ngn: double (nullable = true)
 |-- default_30d_count: long (nullable = true)
 |-- default_90d_count: long (nullable = true)
 |-- default_30d_rate_pct: double (nullable = true)
 |-- default_90d_rate_pct: double (nullable = true)
 |-- default_30d_contribution_pct: double (nullable = true)
 |-- default_90d_contribution_pct: double (nullable = true)
 |-- segment_label: string (nullable = true)

Number of behavioural segments: 2
Required segmentation fields verified.


cluster,segment_label,customers,transactions,exposure_ngn,exposure_share_pct,observed_default_30d_pct,observed_default_90d_pct,default_90d_contribution_pct
0,Low-Engagement / Low-Exposure,150376,204666,8.481038134E9,8.48,4.46,7.31,9.35
1,Active / Higher-Exposure,482980,1795334,9.147498048608E10,91.52,5.06,8.08,90.65


cluster,segment_label,customers,exposure_ngn,exposure_share_pct,observed_default_90d_pct,default_90d_contribution_pct,management_priority,recommended_action
0,Low-Engagement / Low-Exposure,150376,8.481038134E9,8.48,7.31,9.35,Targeted monitoring,Maintain targeted monitoring while avoiding unnecessary credit restriction on a relatively small exposure segment.
1,Active / Higher-Exposure,482980,9.147498048608E10,91.52,8.08,90.65,High portfolio priority,"Prioritise exposure monitoring, repeat-borrowing controls and early-warning surveillance because this segment contains most portfolio exposure."



CUSTOMER SEGMENTATION EXECUTIVE TAKEAWAY
Active / Higher-Exposure represents 91.52% of segment exposure.
It contributes 90.65% of observed 90-day defaults.
Observed 90-day default rate: 8.08%.

Management implication: prioritise exposure monitoring, repeat-borrowing controls and early-warning surveillance within the Active / Higher-Exposure segment.


## 4.8 Early Warning Indicators

The temporal deterioration analysis provides an early-warning perspective on
borrower risk by comparing observed application-level default rates for customers
with and without recent behavioural deterioration signals.

The indicators are derived from Home Credit historical repayment and credit
activity data and are used as risk-intelligence signals rather than as a separate
default prediction model.

The executive objective is to identify which observable behavioural changes are
most strongly associated with elevated default incidence and therefore deserve
greater monitoring attention.

### Management interpretation framework

**Early-warning signal → observed risk lift → monitoring implication → management action**

The strongest indicators are prioritised for surveillance, while recognising that
these relationships are observational and specific to the Home Credit population.

These indicators can support:

- early-warning monitoring
- collections prioritisation
- repayment-behaviour surveillance
- customer risk review
- deterioration-triggered intervention

The temporal analysis does not establish causality and does not imply that these
signals should be interpreted as standalone Probability of Default estimates.

In [0]:
# ============================================================
# 4.8 Early Warning Indicators
# ============================================================

from pyspark.sql import functions as F


# ------------------------------------------------------------
# 4.8.1 Load frozen temporal risk-intelligence summary
# ------------------------------------------------------------

TEMPORAL_SUMMARY_PATH = (
    "/Volumes/workspace/default/home_credit_raw/"
    "home_credit_temporal_risk_summary"
)

temporal_risk_summary = (
    spark.read
    .format("delta")
    .load(TEMPORAL_SUMMARY_PATH)
)

print("Temporal risk-intelligence summary loaded.")
print("Rows:", f"{temporal_risk_summary.count():,}")

temporal_risk_summary.printSchema()


# ------------------------------------------------------------
# 4.8.2 Remove overall reference row
# ------------------------------------------------------------

early_warning = (
    temporal_risk_summary
    .filter(
        F.col("feature") != "TARGET"
    )
    .filter(
        F.col("default_rate_lift").isNotNull()
    )
)


# ------------------------------------------------------------
# 4.8.3 Rank indicators by observed default-rate lift
# ------------------------------------------------------------

early_warning_ranked = (
    early_warning
    .withColumn(
        "risk_lift_x",
        F.round(
            F.col("default_rate_lift"),
            2
        )
    )
    .withColumn(
        "flagged_share_pct",
        F.round(
            F.col("flagged_portfolio_share") * 100,
            2
        )
    )
    .withColumn(
        "flagged_default_rate_pct",
        F.round(
            F.col("flagged_default_rate") * 100,
            2
        )
    )
    .withColumn(
        "unflagged_default_rate_pct",
        F.round(
            F.col("unflagged_default_rate") * 100,
            2
        )
    )
    .orderBy(
        F.desc("default_rate_lift")
    )
)


# ------------------------------------------------------------
# 4.8.4 Executive early-warning table
# ------------------------------------------------------------

early_warning_executive = (
    early_warning_ranked
    .select(
        "indicator",
        "flagged_applications",
        "flagged_share_pct",
        "flagged_default_rate_pct",
        "unflagged_default_rate_pct",
        "risk_lift_x"
    )
)

display(early_warning_executive)


# ------------------------------------------------------------
# 4.8.5 Identify priority indicators
# ------------------------------------------------------------

priority_indicators = (
    early_warning_ranked
    .filter(
        F.col("default_rate_lift") > 1
    )
    .select(
        "indicator",
        "risk_lift_x",
        "flagged_share_pct"
    )
    .orderBy(
        F.desc("default_rate_lift")
    )
)

print("=" * 70)
print("PRIORITY EARLY-WARNING INDICATORS")
print("=" * 70)

display(priority_indicators)


# ------------------------------------------------------------
# 4.8.6 Management interpretation
# ------------------------------------------------------------

management_actions = (
    early_warning_ranked
    .withColumn(
        "monitoring_priority",
        F.when(
            F.col("default_rate_lift") >= 2.0,
            F.lit("Very High")
        )
        .when(
            F.col("default_rate_lift") >= 1.5,
            F.lit("High")
        )
        .when(
            F.col("default_rate_lift") > 1.0,
            F.lit("Moderate")
        )
        .otherwise(
            F.lit("Low")
        )
    )
    .withColumn(
        "management_action",
        F.when(
            F.col("default_rate_lift") >= 2.0,
            F.lit(
                "Prioritise for early-warning surveillance "
                "and targeted intervention."
            )
        )
        .when(
            F.col("default_rate_lift") >= 1.5,
            F.lit(
                "Include in enhanced monitoring and "
                "risk-review workflows."
            )
        )
        .when(
            F.col("default_rate_lift") > 1.0,
            F.lit(
                "Monitor as a supporting deterioration signal."
            )
        )
        .otherwise(
            F.lit(
                "Retain as contextual risk information."
            )
        )
    )
    .select(
        "indicator",
        "risk_lift_x",
        "monitoring_priority",
        "management_action"
    )
    .orderBy(
        F.desc("default_rate_lift")
    )
)

display(management_actions)


# ------------------------------------------------------------
# 4.8.7 Executive takeaway
# ------------------------------------------------------------

top_indicators = (
    early_warning_ranked
    .orderBy(F.desc("default_rate_lift"))
    .limit(5)
    .collect()
)

print("\n" + "=" * 70)
print("EARLY-WARNING EXECUTIVE TAKEAWAY")
print("=" * 70)

for rank, row in enumerate(top_indicators, start=1):

    print(
        f"{rank}. {row['indicator']}: "
        f"{row['default_rate_lift']:.2f}x "
        f"default-rate lift"
    )

print(
    "\nManagement implication: "
    "recent delinquency, repayment deterioration and increasing "
    "credit utilisation should be treated as surveillance signals "
    "within the broader risk-management framework."
)

print(
    "\nImportant limitation: "
    "these lifts are observational relationships within the "
    "Home Credit population and are not causal estimates or "
    "standalone Probability of Default measures."
)

Temporal risk-intelligence summary loaded.
Rows: 9
root
 |-- indicator: string (nullable = true)
 |-- feature: string (nullable = true)
 |-- flagged_applications: long (nullable = true)
 |-- flagged_portfolio_share: double (nullable = true)
 |-- flagged_default_rate: double (nullable = true)
 |-- unflagged_default_rate: double (nullable = true)
 |-- default_rate_lift: double (nullable = true)



indicator,flagged_applications,flagged_share_pct,flagged_default_rate_pct,unflagged_default_rate_pct,risk_lift_x
Recent POS DPD,2111,0.69,16.82,8.01,2.1
Recent credit-card DPD,1172,0.38,14.93,8.05,1.86
Credit-card utilization increase,24190,7.87,13.78,7.59,1.82
Recent installment late payment,25181,8.19,13.21,7.61,1.73
Any recent delinquency,32468,10.56,12.99,7.49,1.73
Recent deterioration,46601,15.15,12.58,7.27,1.73
Recent bureau delinquency,7318,2.38,13.2,7.95,1.66
Recent bureau severe delinquency,1256,0.41,12.5,8.05,1.55


PRIORITY EARLY-WARNING INDICATORS


indicator,risk_lift_x,flagged_share_pct
Recent POS DPD,2.1,0.69
Recent credit-card DPD,1.86,0.38
Credit-card utilization increase,1.82,7.87
Recent installment late payment,1.73,8.19
Any recent delinquency,1.73,10.56
Recent deterioration,1.73,15.15
Recent bureau delinquency,1.66,2.38
Recent bureau severe delinquency,1.55,0.41


indicator,risk_lift_x,monitoring_priority,management_action
Recent POS DPD,2.1,Very High,Prioritise for early-warning surveillance and targeted intervention.
Recent credit-card DPD,1.86,High,Include in enhanced monitoring and risk-review workflows.
Credit-card utilization increase,1.82,High,Include in enhanced monitoring and risk-review workflows.
Recent installment late payment,1.73,High,Include in enhanced monitoring and risk-review workflows.
Any recent delinquency,1.73,High,Include in enhanced monitoring and risk-review workflows.
Recent deterioration,1.73,High,Include in enhanced monitoring and risk-review workflows.
Recent bureau delinquency,1.66,High,Include in enhanced monitoring and risk-review workflows.
Recent bureau severe delinquency,1.55,High,Include in enhanced monitoring and risk-review workflows.



EARLY-WARNING EXECUTIVE TAKEAWAY
1. Recent POS DPD: 2.10x default-rate lift
2. Recent credit-card DPD: 1.86x default-rate lift
3. Credit-card utilization increase: 1.82x default-rate lift
4. Recent installment late payment: 1.73x default-rate lift
5. Any recent delinquency: 1.73x default-rate lift

Management implication: recent delinquency, repayment deterioration and increasing credit utilisation should be treated as surveillance signals within the broader risk-management framework.

Important limitation: these lifts are observational relationships within the Home Credit population and are not causal estimates or standalone Probability of Default measures.


## 4.9 Model Performance → Business Value

The selected BNPL Random Forest model is translated from statistical performance
into an operational risk-screening view.

Model performance alone does not establish business value. The selected operating
threshold determines how the portfolio is separated into transactions requiring
risk intervention and transactions that can proceed through standard treatment.

The analysis therefore evaluates:

- proportion of transactions flagged for intervention
- observed default capture among flagged transactions
- precision of the intervention population
- recall of observed defaults
- false-positive trade-off
- operational implications for approval, monitoring and collections

The threshold was selected using the 2023 validation period and then applied
unchanged to the 2024 out-of-time population.

This section is intended to answer:

**"What does the selected model actually enable management to do?"**

The results are interpreted as decision-support evidence, not as a causal estimate
of losses avoided or defaults prevented.

In [0]:
# ============================================================
# 4.9 Model Performance → Business Value
# ============================================================

from pyspark.sql import functions as F


# ------------------------------------------------------------
# 4.9.1 Load frozen 2024 out-of-time predictions
# ------------------------------------------------------------

FINAL_TEST_PATH = (
    "/Volumes/workspace/default/bnpl_raw/"
    "final_bnpl_test_predictions"
)

business_value_predictions = (
    spark.read
    .format("delta")
    .load(FINAL_TEST_PATH)
)

print("Frozen 2024 out-of-time predictions loaded.")
print(
    f"Transactions: "
    f"{business_value_predictions.count():,}"
)

business_value_predictions.printSchema()


# ------------------------------------------------------------
# 4.9.2 Validate required fields
# ------------------------------------------------------------

required_cols = [
    "transaction_id",
    "customer_id",
    "label",
    "default_probability",
    "risk_prediction"
]

missing_cols = [
    c for c in required_cols
    if c not in business_value_predictions.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required prediction columns: {missing_cols}"
    )

print("Required prediction fields verified.")


# ------------------------------------------------------------
# 4.9.3 Calculate operational screening metrics
# ------------------------------------------------------------

total_transactions = (
    business_value_predictions.count()
)

total_defaults = (
    business_value_predictions
    .agg(F.sum("label").alias("defaults"))
    .collect()[0]["defaults"]
)

flagged_population = (
    business_value_predictions
    .filter(F.col("risk_prediction") == 1)
)

flagged_transactions = flagged_population.count()

flagged_defaults = (
    flagged_population
    .agg(F.sum("label").alias("defaults"))
    .collect()[0]["defaults"]
)

unflagged_population = (
    business_value_predictions
    .filter(F.col("risk_prediction") == 0)
)

unflagged_transactions = unflagged_population.count()

unflagged_defaults = (
    unflagged_population
    .agg(F.sum("label").alias("defaults"))
    .collect()[0]["defaults"]
)


# ------------------------------------------------------------
# 4.9.4 Derive business-facing metrics
# ------------------------------------------------------------

flagged_share_pct = (
    flagged_transactions
    / total_transactions
    * 100
)

default_capture_pct = (
    flagged_defaults
    / total_defaults
    * 100
)

precision_pct = (
    flagged_defaults
    / flagged_transactions
    * 100
    if flagged_transactions > 0
    else 0
)

false_positive_transactions = (
    flagged_transactions - flagged_defaults
)

false_positive_rate_pct = (
    false_positive_transactions
    / (
        total_transactions - total_defaults
    )
    * 100
    if total_transactions > total_defaults
    else 0
)

unflagged_default_rate_pct = (
    unflagged_defaults
    / unflagged_transactions
    * 100
    if unflagged_transactions > 0
    else 0
)


# ------------------------------------------------------------
# 4.9.5 Executive business-value scorecard
# ------------------------------------------------------------

business_value_rows = [
    (
        "Transactions reviewed",
        float(total_transactions),
        "Transactions",
        "Total 2024 out-of-time transactions evaluated."
    ),
    (
        "Intervention population",
        float(flagged_share_pct),
        "%",
        "Share of transactions flagged for risk intervention."
    ),
    (
        "Observed defaults captured",
        float(default_capture_pct),
        "%",
        "Share of observed 90-day defaults occurring within the flagged population."
    ),
    (
        "Intervention precision",
        float(precision_pct),
        "%",
        "Share of flagged transactions that subsequently recorded an observed default."
    ),
    (
        "False-positive rate",
        float(false_positive_rate_pct),
        "%",
        "Share of observed non-default transactions that were nevertheless flagged."
    ),
    (
        "Residual default rate",
        float(unflagged_default_rate_pct),
        "%",
        "Observed 90-day default rate among transactions not flagged."
    )
]

business_value_schema = """
metric STRING,
value DOUBLE,
unit STRING,
business_interpretation STRING
"""

business_value_scorecard = (
    spark.createDataFrame(
        business_value_rows,
        schema=business_value_schema
    )
)

display(
    business_value_scorecard
)


# ------------------------------------------------------------
# 4.9.6 Risk-screening population comparison
# ------------------------------------------------------------

screening_comparison = (
    business_value_predictions
    .withColumn(
        "screening_group",
        F.when(
            F.col("risk_prediction") == 1,
            F.lit("Flagged for Intervention")
        )
        .otherwise(
            F.lit("Standard Treatment")
        )
    )
    .groupBy("screening_group")
    .agg(
        F.count("*").alias("transactions"),
        F.sum("label").alias("observed_defaults"),
        F.avg("label").alias("observed_default_rate"),
        F.avg("default_probability").alias(
            "average_predicted_probability"
        )
    )
    .withColumn(
        "transaction_share_pct",
        F.col("transactions")
        / F.lit(float(total_transactions))
        * 100
    )
    .withColumn(
        "default_share_pct",
        F.col("observed_defaults")
        / F.lit(float(total_defaults))
        * 100
    )
    .select(
        "screening_group",
        "transactions",
        F.round(
            "transaction_share_pct",
            2
        ).alias("transaction_share_pct"),
        "observed_defaults",
        F.round(
            "default_share_pct",
            2
        ).alias("default_capture_share_pct"),
        F.round(
            F.col("observed_default_rate") * 100,
            2
        ).alias("observed_default_rate_pct"),
        F.round(
            F.col("average_predicted_probability") * 100,
            2
        ).alias("average_predicted_probability_pct")
    )
)

display(screening_comparison)


# ------------------------------------------------------------
# 4.9.7 Executive interpretation
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("MODEL-TO-BUSINESS VALUE TRANSLATION")
print("=" * 75)

print(
    f"Intervention population: "
    f"{flagged_share_pct:.2f}% of transactions"
)

print(
    f"Observed 90-day default capture: "
    f"{default_capture_pct:.2f}%"
)

print(
    f"Intervention precision: "
    f"{precision_pct:.2f}%"
)

print(
    f"Residual observed default rate outside intervention population: "
    f"{unflagged_default_rate_pct:.2f}%"
)

print(
    "\nManagement implication:"
)

print(
    "The selected threshold creates a targeted intervention population "
    "that captures a large proportion of observed defaults while "
    "leaving the remaining transactions available for standard treatment."
)

print(
    "\nImportant interpretation:"
)

print(
    "Default capture represents observed historical outcomes in the "
    "2024 out-of-time sample. It should not be described as defaults "
    "prevented or losses avoided."
)

Frozen 2024 out-of-time predictions loaded.
Transactions: 666,246
root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- purchase_date: timestamp (nullable = true)
 |-- principal_ngn: double (nullable = true)
 |-- label: double (nullable = true)
 |-- default_probability: double (nullable = true)
 |-- risk_prediction: integer (nullable = true)

Required prediction fields verified.


metric,value,unit,business_interpretation
Transactions reviewed,666246.0,Transactions,Total 2024 out-of-time transactions evaluated.
Intervention population,18.636659732291076,%,Share of transactions flagged for risk intervention.
Observed defaults captured,95.1303695366723,%,Share of observed 90-day defaults occurring within the flagged population.
Intervention precision,40.84370922796901,%,Share of flagged transactions that subsequently recorded an observed default.
False-positive rate,11.983632875210462,%,Share of observed non-default transactions that were nevertheless flagged.
Residual default rate,0.4788961038961039,%,Observed 90-day default rate among transactions not flagged.


screening_group,transactions,transaction_share_pct,observed_defaults,default_capture_share_pct,observed_default_rate_pct,average_predicted_probability_pct
Flagged for Intervention,124166,18.64,50714.0,95.13,40.84,21.85
Standard Treatment,542080,81.36,2596.0,4.87,0.48,4.91



MODEL-TO-BUSINESS VALUE TRANSLATION
Intervention population: 18.64% of transactions
Observed 90-day default capture: 95.13%
Intervention precision: 40.84%
Residual observed default rate outside intervention population: 0.48%

Management implication:
The selected threshold creates a targeted intervention population that captures a large proportion of observed defaults while leaving the remaining transactions available for standard treatment.

Important interpretation:
Default capture represents observed historical outcomes in the 2024 out-of-time sample. It should not be described as defaults prevented or losses avoided.


## 4.10 Expected Loss Economics

Expected Loss translates modelled credit risk into an exposure-based financial
measure for portfolio management.

The framework is:

**Expected Loss = PD × EAD × LGD**

The portfolio-level expected-loss values are consumed from the frozen Portfolio
Risk Engine outputs. Risk-band expected-loss values are taken from the persisted
stress risk-band layer under the Baseline scenario.

The analysis focuses on:

- total portfolio exposure
- PD-weighted exposure
- baseline expected loss
- expected loss as a percentage of EAD
- expected loss concentration by risk band
- Very High risk-band contribution to expected loss

Expected Loss is interpreted as a scenario-based risk measure. It is not an
accounting provision, an IFRS-compliant expected credit loss estimate, or a
realised loss forecast.

In [0]:
# ============================================================
# 4.10 Expected Loss Economics
# ============================================================

from pyspark.sql import functions as F


# ------------------------------------------------------------
# 4.10.1 Validate the actual persisted portfolio-risk schema
# ------------------------------------------------------------

required_portfolio_cols = [
    "risk_band",
    "portfolio_pd",
    "ead_ngn"
]

missing_portfolio_cols = [
    col_name
    for col_name in required_portfolio_cols
    if col_name not in portfolio_risk.columns
]

if missing_portfolio_cols:
    raise ValueError(
        "Missing required portfolio-risk columns: "
        f"{missing_portfolio_cols}"
    )

print("Required portfolio-risk fields verified.")


# ------------------------------------------------------------
# 4.10.2 Validate expected-loss output
# ------------------------------------------------------------

required_el_cols = [
    "lgd_scenario",
    "lgd",
    "pd_ead_exposure_ngn",
    "expected_loss_ngn",
    "expected_loss_rate_of_ead"
]

missing_el_cols = [
    col_name
    for col_name in required_el_cols
    if col_name not in expected_loss.columns
]

if missing_el_cols:
    raise ValueError(
        "Missing required expected-loss columns: "
        f"{missing_el_cols}"
    )


# ------------------------------------------------------------
# 4.10.3 Extract baseline expected-loss metrics
# ------------------------------------------------------------

baseline_row = (
    expected_loss
    .filter(
        F.col("lgd_scenario") == "Baseline"
    )
    .select(
        "lgd",
        "pd_ead_exposure_ngn",
        "expected_loss_ngn",
        "expected_loss_rate_of_ead"
    )
    .first()
)

if baseline_row is None:
    raise ValueError(
        "Baseline expected-loss scenario was not found."
    )


baseline_lgd = float(
    baseline_row["lgd"]
)

baseline_pd_ead = float(
    baseline_row["pd_ead_exposure_ngn"]
)

baseline_expected_loss = float(
    baseline_row["expected_loss_ngn"]
)

baseline_el_rate = float(
    baseline_row["expected_loss_rate_of_ead"]
)


# ------------------------------------------------------------
# 4.10.4 Calculate total portfolio EAD
# ------------------------------------------------------------

total_ead = float(
    portfolio_risk
    .agg(
        F.sum("ead_ngn").alias(
            "total_ead_ngn"
        )
    )
    .first()["total_ead_ngn"]
)


# ------------------------------------------------------------
# 4.10.5 Executive Expected Loss Scorecard
# ------------------------------------------------------------

expected_loss_rows = [
    (
        "Total Portfolio EAD",
        total_ead,
        "NGN",
        "Total principal-based exposure represented in the portfolio."
    ),
    (
        "PD-Weighted Exposure",
        baseline_pd_ead,
        "NGN",
        "Exposure-weighted risk amount before applying LGD."
    ),
    (
        "Baseline LGD",
        baseline_lgd * 100,
        "%",
        "Baseline loss-severity assumption used in the risk engine."
    ),
    (
        "Baseline Expected Loss",
        baseline_expected_loss,
        "NGN",
        "Scenario-based expected loss under baseline conditions."
    ),
    (
        "Expected Loss / EAD",
        baseline_el_rate * 100,
        "%",
        "Baseline expected loss expressed relative to total exposure."
    )
]


expected_loss_scorecard = spark.createDataFrame(
    expected_loss_rows,
    schema="""
        metric STRING,
        value DOUBLE,
        unit STRING,
        business_interpretation STRING
    """
)

display(expected_loss_scorecard)


# ------------------------------------------------------------
# 4.10.6 Calculate risk-band Expected Loss
# ------------------------------------------------------------
#
# Persisted portfolio-risk field:
#     portfolio_pd
#
# Exposure:
#     ead_ngn
#
# Formula:
#     Expected Loss = portfolio_pd × EAD × baseline LGD
#
# This is calculated directly from the frozen transaction-level
# portfolio risk layer.
# ------------------------------------------------------------

risk_band_el_raw = (
    portfolio_risk
    .withColumn(
        "pd_weighted_exposure_ngn",
        F.col("portfolio_pd")
        * F.col("ead_ngn")
    )
    .withColumn(
        "expected_loss_ngn",
        F.col("portfolio_pd")
        * F.col("ead_ngn")
        * F.lit(baseline_lgd)
    )
    .groupBy("risk_band")
    .agg(
        F.count("*").alias(
            "transaction_count"
        ),
        F.sum("ead_ngn").alias(
            "ead_ngn"
        ),
        F.avg("portfolio_pd").alias(
            "average_pd"
        ),
        F.sum(
            "pd_weighted_exposure_ngn"
        ).alias(
            "pd_weighted_exposure_ngn"
        ),
        F.sum(
            "expected_loss_ngn"
        ).alias(
            "expected_loss_ngn"
        )
    )
)


# ------------------------------------------------------------
# 4.10.7 Calculate portfolio shares
# ------------------------------------------------------------

risk_band_el = (
    risk_band_el_raw
    .withColumn(
        "ead_share_pct",
        F.when(
            F.lit(total_ead) > 0,
            F.col("ead_ngn")
            / F.lit(total_ead)
            * 100
        )
        .otherwise(
            F.lit(0.0)
        )
    )
    .withColumn(
        "expected_loss_rate_of_ead_pct",
        F.when(
            F.col("ead_ngn") > 0,
            F.col("expected_loss_ngn")
            / F.col("ead_ngn")
            * 100
        )
        .otherwise(
            F.lit(0.0)
        )
    )
    .withColumn(
        "expected_loss_share_pct",
        F.when(
            F.lit(baseline_expected_loss) > 0,
            F.col("expected_loss_ngn")
            / F.lit(baseline_expected_loss)
            * 100
        )
        .otherwise(
            F.lit(0.0)
        )
    )
    .withColumn(
        "risk_band_order",
        F.when(
            F.col("risk_band") == "Very Low",
            1
        )
        .when(
            F.col("risk_band") == "Low",
            2
        )
        .when(
            F.col("risk_band") == "Moderate",
            3
        )
        .when(
            F.col("risk_band") == "High",
            4
        )
        .when(
            F.col("risk_band") == "Very High",
            5
        )
        .otherwise(99)
    )
)


# ------------------------------------------------------------
# 4.10.8 Display risk-band economics
# ------------------------------------------------------------

risk_band_el_display = (
    risk_band_el
    .orderBy("risk_band_order")
    .select(
        "risk_band",
        "transaction_count",
        F.round(
            F.col("ead_ngn"),
            2
        ).alias(
            "ead_ngn"
        ),
        F.round(
            F.col("ead_share_pct"),
            2
        ).alias(
            "ead_share_pct"
        ),
        F.round(
            F.col("average_pd") * 100,
            2
        ).alias(
            "average_pd_pct"
        ),
        F.round(
            F.col("pd_weighted_exposure_ngn"),
            2
        ).alias(
            "pd_weighted_exposure_ngn"
        ),
        F.round(
            F.col("expected_loss_ngn"),
            2
        ).alias(
            "expected_loss_ngn"
        ),
        F.round(
            F.col("expected_loss_rate_of_ead_pct"),
            2
        ).alias(
            "expected_loss_rate_of_ead_pct"
        ),
        F.round(
            F.col("expected_loss_share_pct"),
            2
        ).alias(
            "expected_loss_share_pct"
        )
    )
)

display(risk_band_el_display)


# ------------------------------------------------------------
# 4.10.9 Extract Very High risk-band economics
# ------------------------------------------------------------

very_high_row = (
    risk_band_el
    .filter(
        F.col("risk_band") == "Very High"
    )
    .first()
)

if very_high_row is None:
    raise ValueError(
        "Very High risk band was not found."
    )


vh_ead_share = float(
    very_high_row["ead_share_pct"]
)

vh_pd_weighted_exposure = float(
    very_high_row[
        "pd_weighted_exposure_ngn"
    ]
)

vh_expected_loss = float(
    very_high_row[
        "expected_loss_ngn"
    ]
)

vh_expected_loss_share = float(
    very_high_row[
        "expected_loss_share_pct"
    ]
)


# ------------------------------------------------------------
# 4.10.10 Very High risk concentration ratio
# ------------------------------------------------------------

vh_pd_ead_share = (
    vh_pd_weighted_exposure
    / baseline_pd_ead
    * 100
    if baseline_pd_ead > 0
    else 0.0
)

risk_concentration_ratio = (
    vh_pd_ead_share
    / vh_ead_share
    if vh_ead_share > 0
    else 0.0
)


# ------------------------------------------------------------
# 4.10.11 Executive interpretation
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("EXPECTED LOSS ECONOMICS")
print("=" * 75)

print(
    f"Total Portfolio EAD: "
    f"₦{total_ead:,.2f}"
)

print(
    f"PD-Weighted Exposure: "
    f"₦{baseline_pd_ead:,.2f}"
)

print(
    f"Baseline LGD: "
    f"{baseline_lgd * 100:.1f}%"
)

print(
    f"Baseline Expected Loss: "
    f"₦{baseline_expected_loss:,.2f}"
)

print(
    f"Expected Loss / EAD: "
    f"{baseline_el_rate * 100:.2f}%"
)

print(
    f"Very High Risk EAD Share: "
    f"{vh_ead_share:.2f}%"
)

print(
    f"Very High PD-Weighted Exposure Share: "
    f"{vh_pd_ead_share:.2f}%"
)

print(
    f"Very High Expected Loss: "
    f"₦{vh_expected_loss:,.2f}"
)

print(
    f"Very High Expected Loss Share: "
    f"{vh_expected_loss_share:.2f}%"
)

print(
    f"Risk Concentration Ratio: "
    f"{risk_concentration_ratio:.2f}x"
)


# ------------------------------------------------------------
# 4.10.12 Management interpretation
# ------------------------------------------------------------

print("\nManagement implication:")

print(
    "The Very High risk band carries a disproportionately large share "
    "of modelled risk and expected loss relative to its share of total "
    "portfolio exposure. Management attention should therefore focus "
    "on exposure controls, monitoring intensity and collections "
    "prioritisation within the highest-risk population."
)

print("\nMethodological caution:")

print(
    "Expected Loss is scenario-based and depends on the stated LGD "
    "assumption. It is not an accounting provision, an IFRS-compliant "
    "expected credit loss estimate, or an empirical recovery forecast."
)

Required portfolio-risk fields verified.


metric,value,unit,business_interpretation
Total Portfolio EAD,3.3317913042690628E10,NGN,Total principal-based exposure represented in the portfolio.
PD-Weighted Exposure,3.289674922513504E9,NGN,Exposure-weighted risk amount before applying LGD.
Baseline LGD,40.0,%,Baseline loss-severity assumption used in the risk engine.
Baseline Expected Loss,1.3158699690054088E9,NGN,Scenario-based expected loss under baseline conditions.
Expected Loss / EAD,3.949436950986159,%,Baseline expected loss expressed relative to total exposure.


risk_band,transaction_count,ead_ngn,ead_share_pct,average_pd_pct,pd_weighted_exposure_ngn,expected_loss_ngn,expected_loss_rate_of_ead_pct,expected_loss_share_pct
Very Low,133214,4.99694278068E9,15.0,2.36,1.1812810701E8,4.72512428E7,0.95,3.59
Low,133296,5.31099042872E9,15.94,3.47,1.8596393432E8,7.438557373E7,1.4,5.65
Moderate,133206,8.16018938335E9,24.49,5.17,4.2584300628E8,1.7033720251E8,2.09,12.94
High,133290,6.46276671092E9,19.4,7.88,5.1580579967E8,2.0632231987E8,3.19,15.68
Very High,133240,8.38702373902E9,25.17,21.12,2.04393407522E9,8.1757363009E8,9.75,62.13



EXPECTED LOSS ECONOMICS
Total Portfolio EAD: ₦33,317,913,042.69
PD-Weighted Exposure: ₦3,289,674,922.51
Baseline LGD: 40.0%
Baseline Expected Loss: ₦1,315,869,969.01
Expected Loss / EAD: 3.95%
Very High Risk EAD Share: 25.17%
Very High PD-Weighted Exposure Share: 62.13%
Very High Expected Loss: ₦817,573,630.09
Very High Expected Loss Share: 62.13%
Risk Concentration Ratio: 2.47x

Management implication:
The Very High risk band carries a disproportionately large share of modelled risk and expected loss relative to its share of total portfolio exposure. Management attention should therefore focus on exposure controls, monitoring intensity and collections prioritisation within the highest-risk population.

Methodological caution:
Expected Loss is scenario-based and depends on the stated LGD assumption. It is not an accounting provision, an IFRS-compliant expected credit loss estimate, or an empirical recovery forecast.


## 4.11 Stress Testing & Management Interpretation

Stress testing evaluates how portfolio expected loss changes under progressively
adverse risk and loss-severity assumptions.

Three scenarios are considered:

- **Baseline:** current portfolio risk conditions and baseline LGD
- **Moderate:** elevated PD and higher LGD
- **Severe:** further elevated PD and higher LGD

The stress framework applies the frozen scenario assumptions from the Portfolio
Risk Engine. It is designed to assess portfolio resilience rather than forecast
a specific future economic outcome.

The analysis focuses on:

- stressed expected loss
- incremental expected loss versus baseline
- expected-loss multiple versus baseline
- severity of deterioration between scenarios
- management implications for capital protection, exposure controls and
  collections capacity

The macroeconomic component provides scenario context only. It does not establish
a causal relationship between individual macroeconomic variables and BNPL default.

In [0]:
# ============================================================
# 4.11 Stress Testing & Management Interpretation
# ============================================================

from pyspark.sql import functions as F

# ------------------------------------------------------------
# 1. Inspect and validate the frozen stress output
# ------------------------------------------------------------

print("Available stress scenarios:")
display(
    stress_summary
    .select("scenario", "scenario_type")
    .orderBy("scenario")
)


# ------------------------------------------------------------
# 2. Identify scenarios robustly from their actual labels
# ------------------------------------------------------------

scenario_lower = F.lower(F.col("scenario"))

baseline_row = (
    stress_summary
    .filter(scenario_lower == "baseline")
    .first()
)

moderate_row = (
    stress_summary
    .filter(scenario_lower.contains("moderate"))
    .first()
)

severe_row = (
    stress_summary
    .filter(scenario_lower.contains("severe"))
    .first()
)

if baseline_row is None:
    raise ValueError("Baseline scenario was not found.")

if moderate_row is None:
    raise ValueError("Moderate macro stress scenario was not found.")

if severe_row is None:
    raise ValueError("Severe macro stress scenario was not found.")


# ------------------------------------------------------------
# 3. Helper to safely retrieve available columns
# ------------------------------------------------------------

def get_value(row, *column_names):
    """
    Return the first available non-null column value from a Row.
    """
    for column_name in column_names:
        if column_name in row.asDict():
            value = row[column_name]
            if value is not None:
                return value

    return None


# ------------------------------------------------------------
# 4. Extract frozen scenario results
# ------------------------------------------------------------

baseline_el = get_value(
    baseline_row,
    "stressed_expected_loss_ngn",
    "expected_loss_ngn"
)

moderate_el = get_value(
    moderate_row,
    "stressed_expected_loss_ngn",
    "expected_loss_ngn"
)

severe_el = get_value(
    severe_row,
    "stressed_expected_loss_ngn",
    "expected_loss_ngn"
)

baseline_pd_multiplier = get_value(
    baseline_row,
    "pd_multiplier"
)

moderate_pd_multiplier = get_value(
    moderate_row,
    "pd_multiplier"
)

severe_pd_multiplier = get_value(
    severe_row,
    "pd_multiplier"
)

baseline_lgd = get_value(
    baseline_row,
    "stressed_lgd_pct",
    "stressed_lgd"
)

moderate_lgd = get_value(
    moderate_row,
    "stressed_lgd_pct",
    "stressed_lgd"
)

severe_lgd = get_value(
    severe_row,
    "stressed_lgd_pct",
    "stressed_lgd"
)


# ------------------------------------------------------------
# 5. Calculate management-level stress indicators
# ------------------------------------------------------------

severe_vs_baseline_multiple = (
    float(severe_el) / float(baseline_el)
    if baseline_el not in [None, 0]
    else None
)

moderate_vs_baseline_multiple = (
    float(moderate_el) / float(baseline_el)
    if baseline_el not in [None, 0]
    else None
)

moderate_incremental_el = (
    float(moderate_el) - float(baseline_el)
    if baseline_el is not None and moderate_el is not None
    else None
)

severe_incremental_el = (
    float(severe_el) - float(baseline_el)
    if baseline_el is not None and severe_el is not None
    else None
)


# ------------------------------------------------------------
# 6. Executive stress summary
# ------------------------------------------------------------

stress_management_summary = spark.createDataFrame(
    [
        (
            "Baseline",
            float(baseline_pd_multiplier),
            float(baseline_lgd),
            float(baseline_el),
            1.00,
            0.00,
            "Current/reference portfolio loss position"
        ),
        (
            "Moderate Macro Stress",
            float(moderate_pd_multiplier),
            float(moderate_lgd),
            float(moderate_el),
            float(moderate_vs_baseline_multiple),
            float(moderate_incremental_el),
            "Prepare for higher credit losses and tighten risk controls"
        ),
        (
            "Severe Macro Stress",
            float(severe_pd_multiplier),
            float(severe_lgd),
            float(severe_el),
            float(severe_vs_baseline_multiple),
            float(severe_incremental_el),
            "Protect capital, restrict incremental exposure and strengthen collections"
        )
    ],
    [
        "scenario",
        "pd_multiplier",
        "lgd_pct",
        "expected_loss_ngn",
        "el_vs_baseline_multiple",
        "incremental_el_ngn",
        "management_interpretation"
    ]
)


display(
    stress_management_summary
    .orderBy(
        F.when(F.col("scenario") == "Baseline", 1)
         .when(F.col("scenario") == "Moderate Macro Stress", 2)
         .when(F.col("scenario") == "Severe Macro Stress", 3)
         .otherwise(99)
    )
)


# ------------------------------------------------------------
# 7. Executive takeaway
# ------------------------------------------------------------

print("STRESS TESTING MANAGEMENT TAKEAWAY")
print("=" * 60)

print(
    f"Baseline expected loss: "
    f"₦{float(baseline_el):,.2f}"
)

print(
    f"Moderate macro stress expected loss: "
    f"₦{float(moderate_el):,.2f}"
)

print(
    f"Severe macro stress expected loss: "
    f"₦{float(severe_el):,.2f}"
)

print(
    f"Severe stress / baseline EL: "
    f"{float(severe_vs_baseline_multiple):.2f}x"
)

print(
    "\nManagement implication:"
)

print(
    "Stress testing should be used to determine the portfolio's "
    "loss-bearing capacity and guide exposure limits, collections "
    "prioritisation and risk-adjusted growth."
)

print(
    "\nMethodological boundary:"
)

print(
    "These are scenario-based sensitivity estimates, not macroeconomic "
    "causal forecasts. The BNPL target is synthetic and the macro panel "
    "contains only 36 monthly observations."
)

Available stress scenarios:


scenario,scenario_type
Baseline,Baseline
Moderate Macro Stress,Macro stress
Severe Macro Stress,Macro stress


scenario,pd_multiplier,lgd_pct,expected_loss_ngn,el_vs_baseline_multiple,incremental_el_ngn,management_interpretation
Baseline,1.0,0.4,1.315869969005415E9,1.0,0.0,Current/reference portfolio loss position
Moderate Macro Stress,1.25,0.55,2.2616515092280593E9,1.7187500000000018,9.457815402226443E8,Prepare for higher credit losses and tighten risk controls
Severe Macro Stress,1.5,0.7,3.4541586686392164E9,2.6250000000000018,2.1382886996338015E9,"Protect capital, restrict incremental exposure and strengthen collections"


STRESS TESTING MANAGEMENT TAKEAWAY
Baseline expected loss: ₦1,315,869,969.01
Moderate macro stress expected loss: ₦2,261,651,509.23
Severe macro stress expected loss: ₦3,454,158,668.64
Severe stress / baseline EL: 2.63x

Management implication:
Stress testing should be used to determine the portfolio's loss-bearing capacity and guide exposure limits, collections prioritisation and risk-adjusted growth.

Methodological boundary:
These are scenario-based sensitivity estimates, not macroeconomic causal forecasts. The BNPL target is synthetic and the macro panel contains only 36 monthly observations.


## 4.12 Strategic Management Recommendations

The portfolio analytics are translated into management actions across five
decision areas: approval, exposure management, monitoring, collections and
stress preparedness.

Recommendations are derived from the combined evidence from:
- BNPL model-derived risk bands
- portfolio exposure concentration
- customer behavioural segments
- temporal early-warning indicators
- expected-loss analysis
- macro stress scenarios

The recommendations are designed as risk-management actions rather than
automatic lending decisions.

### Executive interpretation framework

**Analytical finding → Risk implication → Management action → Expected business value**

The objective is to protect portfolio quality while allowing risk-adjusted
growth in lower-risk portions of the portfolio.

Recommendations should not be interpreted as causal or regulatory requirements.
They are analytical decision-support recommendations based on the project's
synthetic BNPL portfolio and scenario assumptions.

In [0]:
# ============================================================
# 4.12 Strategic Management Recommendations
# ============================================================

from pyspark.sql import functions as F


# ------------------------------------------------------------
# 1. Portfolio-level decision signals
# ------------------------------------------------------------

total_ead = (
    portfolio_risk
    .agg(F.sum("ead_ngn").alias("total_ead"))
    .first()["total_ead"]
)

exposure_weighted_pd = (
    portfolio_risk
    .agg(
        (
            F.sum(F.col("portfolio_pd") * F.col("ead_ngn"))
            / F.sum("ead_ngn")
        ).alias("exposure_weighted_pd")
    )
    .first()["exposure_weighted_pd"]
)


# ------------------------------------------------------------
# 2. Very High risk concentration
# ------------------------------------------------------------

vh_row = (
    risk_band_concentration
    .filter(F.col("risk_band") == "Very High")
    .first()
)

if vh_row is None:
    raise ValueError("Very High risk band was not found.")

vh_ead_share = float(vh_row["ead_share_pct"])


# ------------------------------------------------------------
# 3. Build the management recommendation matrix
# ------------------------------------------------------------

recommendations = [
    (
        "Approval & Risk-Based Growth",
        "Very Low / Low",
        "Lower-risk bands provide the appropriate pool for controlled "
        "risk-adjusted growth.",
        "Prioritise lower-risk customers for repeat borrowing and "
        "selective credit-line expansion, subject to ongoing repayment performance.",
        "Support portfolio growth while maintaining risk discipline."
    ),
    (
        "Exposure Management",
        "High / Very High",
        f"The Very High risk band represents approximately "
        f"{vh_ead_share:.2f}% of portfolio EAD.",
        "Restrict incremental exposure for the highest-risk borrowers "
        "and apply tighter limits to repeat borrowing.",
        "Reduce concentration of exposure in the highest-risk population."
    ),
    (
        "Enhanced Monitoring",
        "Moderate / High / Very High",
        "Higher-risk bands require greater monitoring intensity than "
        "standard-risk customers.",
        "Use differentiated monitoring frequency, with enhanced "
        "surveillance for deteriorating repayment behaviour.",
        "Identify deterioration earlier and reduce avoidable roll-forward risk."
    ),
    (
        "Collections Prioritisation",
        "Very High",
        "The highest-risk band contributes disproportionately to "
        "PD-weighted portfolio exposure.",
        "Prioritise collections and intervention resources toward "
        "the highest-risk exposures before broad portfolio escalation.",
        "Improve collections efficiency by focusing resources where "
        "potential loss concentration is greatest."
    ),
    (
        "Stress Preparedness",
        "Portfolio-wide",
        f"Severe macro stress increases expected loss to approximately "
        f"{float(severe_el):,.0f} NGN, representing "
        f"{float(severe_vs_baseline_multiple):.2f}x baseline expected loss.",
        "Use moderate and severe scenarios to establish contingency "
        "triggers for exposure growth, collections capacity and loss absorption.",
        "Improve preparedness for adverse portfolio conditions."
    )
]


recommendation_schema = [
    "decision_area",
    "priority_risk_group",
    "analytical_evidence",
    "recommended_action",
    "expected_business_value"
]

recommendation_df = spark.createDataFrame(
    recommendations,
    recommendation_schema
)

display(recommendation_df)


# ------------------------------------------------------------
# 4. Executive priority summary
# ------------------------------------------------------------

priority_summary = [
    (
        1,
        "Protect the highest-risk exposure",
        "Very High risk concentration requires the strongest exposure "
        "controls and collections prioritisation."
    ),
    (
        2,
        "Strengthen early-warning monitoring",
        "Increase monitoring intensity as repayment deterioration and "
        "risk-band severity increase."
    ),
    (
        3,
        "Enable risk-adjusted growth",
        "Use lower-risk segments and bands as the primary pool for "
        "controlled portfolio expansion."
    ),
    (
        4,
        "Prepare for adverse scenarios",
        "Use stress-test losses to inform contingency planning and "
        "portfolio risk appetite."
    )
]

priority_df = spark.createDataFrame(
    priority_summary,
    [
        "priority",
        "management_priority",
        "executive_rationale"
    ]
)

display(
    priority_df.orderBy("priority")
)


# ------------------------------------------------------------
# 5. Final management message
# ------------------------------------------------------------

print("STRATEGIC MANAGEMENT MESSAGE")
print("=" * 70)

print(
    "The portfolio should be managed through differentiated risk controls "
    "rather than a uniform lending strategy."
)

print(
    f"\nVery High risk EAD share: {vh_ead_share:.2f}%"
)

print(
    f"Exposure-weighted portfolio PD: "
    f"{float(exposure_weighted_pd) * 100:.2f}%"
)

print(
    f"Baseline expected loss: "
    f"₦{float(baseline_el):,.2f}"
)

print(
    f"Severe stress expected loss: "
    f"₦{float(severe_el):,.2f}"
)

print(
    f"Severe stress / baseline EL: "
    f"{float(severe_vs_baseline_multiple):.2f}x"
)

print(
    "\nRecommended management posture:"
)

print(
    "1. Support controlled growth in lower-risk portions of the portfolio."
)

print(
    "2. Tighten incremental exposure for High and Very High risk borrowers."
)

print(
    "3. Increase monitoring intensity as risk and behavioural deterioration increase."
)

print(
    "4. Prioritise collections resources toward concentrated high-risk exposure."
)

print(
    "5. Use moderate and severe stress scenarios for contingency planning."
)

print(
    "\nResponsible-use boundary:"
)

print(
    "These recommendations are decision-support outputs from a synthetic "
    "BNPL portfolio. They should not be interpreted as automatic approval, "
    "decline or regulatory credit decisions."
)

decision_area,priority_risk_group,analytical_evidence,recommended_action,expected_business_value
Approval & Risk-Based Growth,Very Low / Low,Lower-risk bands provide the appropriate pool for controlled risk-adjusted growth.,"Prioritise lower-risk customers for repeat borrowing and selective credit-line expansion, subject to ongoing repayment performance.",Support portfolio growth while maintaining risk discipline.
Exposure Management,High / Very High,The Very High risk band represents approximately 25.17% of portfolio EAD.,Restrict incremental exposure for the highest-risk borrowers and apply tighter limits to repeat borrowing.,Reduce concentration of exposure in the highest-risk population.
Enhanced Monitoring,Moderate / High / Very High,Higher-risk bands require greater monitoring intensity than standard-risk customers.,"Use differentiated monitoring frequency, with enhanced surveillance for deteriorating repayment behaviour.",Identify deterioration earlier and reduce avoidable roll-forward risk.
Collections Prioritisation,Very High,The highest-risk band contributes disproportionately to PD-weighted portfolio exposure.,Prioritise collections and intervention resources toward the highest-risk exposures before broad portfolio escalation.,Improve collections efficiency by focusing resources where potential loss concentration is greatest.
Stress Preparedness,Portfolio-wide,"Severe macro stress increases expected loss to approximately 3,454,158,669 NGN, representing 2.63x baseline expected loss.","Use moderate and severe scenarios to establish contingency triggers for exposure growth, collections capacity and loss absorption.",Improve preparedness for adverse portfolio conditions.


priority,management_priority,executive_rationale
1,Protect the highest-risk exposure,Very High risk concentration requires the strongest exposure controls and collections prioritisation.
2,Strengthen early-warning monitoring,Increase monitoring intensity as repayment deterioration and risk-band severity increase.
3,Enable risk-adjusted growth,Use lower-risk segments and bands as the primary pool for controlled portfolio expansion.
4,Prepare for adverse scenarios,Use stress-test losses to inform contingency planning and portfolio risk appetite.


STRATEGIC MANAGEMENT MESSAGE
The portfolio should be managed through differentiated risk controls rather than a uniform lending strategy.

Very High risk EAD share: 25.17%
Exposure-weighted portfolio PD: 9.87%
Baseline expected loss: ₦1,315,869,969.01
Severe stress expected loss: ₦3,454,158,668.64
Severe stress / baseline EL: 2.63x

Recommended management posture:
1. Support controlled growth in lower-risk portions of the portfolio.
2. Tighten incremental exposure for High and Very High risk borrowers.
3. Increase monitoring intensity as risk and behavioural deterioration increase.
4. Prioritise collections resources toward concentrated high-risk exposure.
5. Use moderate and severe stress scenarios for contingency planning.

Responsible-use boundary:
These recommendations are decision-support outputs from a synthetic BNPL portfolio. They should not be interpreted as automatic approval, decline or regulatory credit decisions.


## 4.13 Executive Decision Matrix

The executive decision matrix consolidates the portfolio's major analytical
findings into actionable management decisions.

It connects:
- portfolio risk concentration
- customer behavioural profiles
- model-derived risk bands
- expected-loss exposure
- stress-test outcomes

The matrix is designed to support management decisions across growth,
exposure control, monitoring, collections and stress preparedness.

### Decision framework

**Evidence → Risk signal → Management decision → Business objective**

The matrix is a decision-support framework. It does not constitute an
automatic lending policy, regulatory credit grade or causal forecast.

In [0]:
# ============================================================
# 4.13 Executive Decision Matrix
# ============================================================

from pyspark.sql import functions as F


# ------------------------------------------------------------
# 1. Prepare key portfolio indicators
# ------------------------------------------------------------

# Very High risk concentration
vh_row = (
    risk_band_concentration
    .filter(F.col("risk_band") == "Very High")
    .first()
)

if vh_row is None:
    raise ValueError("Very High risk band was not found.")

vh_ead_share = float(vh_row["ead_share_pct"])


# Exposure-weighted PD
exposure_weighted_pd = (
    portfolio_risk
    .agg(
        (
            F.sum(
                F.col("portfolio_pd") * F.col("ead_ngn")
            )
            / F.sum("ead_ngn")
        ).alias("exposure_weighted_pd")
    )
    .first()["exposure_weighted_pd"]
)


# Baseline expected loss
baseline_row = (
    expected_loss
    .filter(F.col("lgd_scenario") == "Baseline")
    .first()
)

if baseline_row is None:
    raise ValueError("Baseline expected-loss scenario was not found.")

baseline_el = float(baseline_row["expected_loss_ngn"])


# Severe stress expected loss
severe_row = (
    stress_summary
    .filter(F.lower(F.col("scenario")).contains("severe"))
    .first()
)

if severe_row is None:
    raise ValueError("Severe macro stress scenario was not found.")

severe_el = float(
    severe_row["stressed_expected_loss_ngn"]
)

severe_vs_baseline_multiple = (
    severe_el / baseline_el
    if baseline_el > 0
    else None
)


# ------------------------------------------------------------
# 2. Build executive decision matrix
# ------------------------------------------------------------

decision_matrix = [
    (
        1,
        "Risk-adjusted growth",
        "Very Low / Low risk bands",
        "Risk concentration is lower in the lower-risk bands.",
        "Support controlled repeat borrowing and selective "
        "credit expansion, subject to repayment performance.",
        "Grow the portfolio without applying uniform risk appetite."
    ),
    (
        2,
        "Exposure control",
        "High / Very High risk bands",
        f"Very High risk represents approximately "
        f"{vh_ead_share:.2f}% of total EAD.",
        "Restrict incremental exposure and apply tighter "
        "borrowing limits to the highest-risk exposures.",
        "Reduce concentration of potential loss."
    ),
    (
        3,
        "Monitoring intensity",
        "Moderate / High / Very High",
        "Risk severity increases the need for differentiated "
        "portfolio surveillance.",
        "Increase monitoring frequency and use behavioural "
        "deterioration indicators as escalation signals.",
        "Detect emerging risk earlier."
    ),
    (
        4,
        "Collections prioritisation",
        "Very High risk",
        "The highest-risk band contributes disproportionately "
        "to portfolio risk concentration.",
        "Prioritise collections and intervention resources "
        "toward the highest-risk exposures.",
        "Focus recovery resources where potential loss is concentrated."
    ),
    (
        5,
        "Stress preparedness",
        "Portfolio-wide",
        f"Severe stress produces approximately "
        f"{severe_vs_baseline_multiple:.2f}x baseline expected loss.",
        "Use stress scenarios to establish contingency actions "
        "for exposure growth, collections capacity and risk appetite.",
        "Improve resilience under adverse conditions."
    )
]


decision_schema = [
    "decision_priority",
    "decision_area",
    "target_population",
    "risk_evidence",
    "management_decision",
    "business_objective"
]


decision_matrix_df = spark.createDataFrame(
    decision_matrix,
    decision_schema
)


display(
    decision_matrix_df
    .orderBy("decision_priority")
)


# ------------------------------------------------------------
# 3. Executive scorecard supporting the decision matrix
# ------------------------------------------------------------

executive_decision_scorecard = spark.createDataFrame(
    [
        (
            "Exposure-Weighted PD",
            float(exposure_weighted_pd) * 100,
            "%",
            "Economically weighted portfolio default risk"
        ),
        (
            "Very High Risk EAD Share",
            vh_ead_share,
            "%",
            "Exposure concentration in the highest-risk band"
        ),
        (
            "Baseline Expected Loss",
            baseline_el,
            "NGN",
            "Scenario-based portfolio expected loss"
        ),
        (
            "Severe Stress Expected Loss",
            severe_el,
            "NGN",
            "Portfolio expected loss under severe stress"
        ),
        (
            "Severe / Baseline EL",
            severe_vs_baseline_multiple,
            "x",
            "Sensitivity of expected loss to severe stress"
        )
    ],
    [
        "executive_metric",
        "value",
        "unit",
        "management_meaning"
    ]
)

display(executive_decision_scorecard)


# ------------------------------------------------------------
# 4. Final executive message
# ------------------------------------------------------------

print("EXECUTIVE DECISION MESSAGE")
print("=" * 70)

print(
    "The portfolio should be managed through differentiated risk "
    "controls rather than a uniform lending strategy."
)

print(
    f"\nExposure-weighted PD: "
    f"{float(exposure_weighted_pd) * 100:.2f}%"
)

print(
    f"Very High risk EAD share: "
    f"{vh_ead_share:.2f}%"
)

print(
    f"Baseline expected loss: "
    f"₦{baseline_el:,.2f}"
)

print(
    f"Severe stress expected loss: "
    f"₦{severe_el:,.2f}"
)

print(
    f"Severe stress / baseline EL: "
    f"{severe_vs_baseline_multiple:.2f}x"
)

print(
    "\nManagement focus:"
)

print(
    "• Enable controlled growth in lower-risk exposures."
)

print(
    "• Control incremental exposure in High and Very High risk bands."
)

print(
    "• Intensify monitoring as risk severity increases."
)

print(
    "• Prioritise collections toward concentrated high-risk exposure."
)

print(
    "• Use stress scenarios to strengthen contingency planning."
)

decision_priority,decision_area,target_population,risk_evidence,management_decision,business_objective
1,Risk-adjusted growth,Very Low / Low risk bands,Risk concentration is lower in the lower-risk bands.,"Support controlled repeat borrowing and selective credit expansion, subject to repayment performance.",Grow the portfolio without applying uniform risk appetite.
2,Exposure control,High / Very High risk bands,Very High risk represents approximately 25.17% of total EAD.,Restrict incremental exposure and apply tighter borrowing limits to the highest-risk exposures.,Reduce concentration of potential loss.
3,Monitoring intensity,Moderate / High / Very High,Risk severity increases the need for differentiated portfolio surveillance.,Increase monitoring frequency and use behavioural deterioration indicators as escalation signals.,Detect emerging risk earlier.
4,Collections prioritisation,Very High risk,The highest-risk band contributes disproportionately to portfolio risk concentration.,Prioritise collections and intervention resources toward the highest-risk exposures.,Focus recovery resources where potential loss is concentrated.
5,Stress preparedness,Portfolio-wide,Severe stress produces approximately 2.63x baseline expected loss.,"Use stress scenarios to establish contingency actions for exposure growth, collections capacity and risk appetite.",Improve resilience under adverse conditions.


executive_metric,value,unit,management_meaning
Exposure-Weighted PD,9.873592377464744,%,Economically weighted portfolio default risk
Very High Risk EAD Share,25.172716335120963,%,Exposure concentration in the highest-risk band
Baseline Expected Loss,1.3158699690054088E9,NGN,Scenario-based portfolio expected loss
Severe Stress Expected Loss,3.4541586686392164E9,NGN,Portfolio expected loss under severe stress
Severe / Baseline EL,2.625000000000014,x,Sensitivity of expected loss to severe stress


EXECUTIVE DECISION MESSAGE
The portfolio should be managed through differentiated risk controls rather than a uniform lending strategy.

Exposure-weighted PD: 9.87%
Very High risk EAD share: 25.17%
Baseline expected loss: ₦1,315,869,969.01
Severe stress expected loss: ₦3,454,158,668.64
Severe stress / baseline EL: 2.63x

Management focus:
• Enable controlled growth in lower-risk exposures.
• Control incremental exposure in High and Very High risk bands.
• Intensify monitoring as risk severity increases.
• Prioritise collections toward concentrated high-risk exposure.
• Use stress scenarios to strengthen contingency planning.


## 4.14 Limitations & Responsible Use

The analytical framework is designed as a decision-support system rather than
a production lending policy or accounting-grade loss engine.

The following limitations define the appropriate interpretation of the results.

### 1. Synthetic BNPL dataset

The primary BNPL dataset is synthetic. Therefore, model performance, default
rates, PD estimates and portfolio loss estimates should not be interpreted as
empirical estimates of the Nigerian BNPL market.

### 2. EAD methodology

Actual outstanding exposure at the point of default is not available in the
BNPL dataset. Principal amount is therefore used as an exposure proxy.

### 3. Scenario-based LGD

Reliable recovery information is unavailable. LGD is therefore represented
through transparent baseline, moderate and severe assumptions rather than
estimated empirically from recoveries.

### 4. Macro stress interpretation

The macro dataset contains only 36 monthly observations. Stress scenarios are
therefore sensitivity analyses rather than causal macroeconomic default
forecasts.

### 5. Home Credit population

Home Credit is a separate traditional-credit reference environment. Its
customers are not merged with the BNPL population, and its model results are
not treated as improvements to the BNPL PD model.

### 6. Risk-band interpretation

Risk bands are empirical distribution-based portfolio tiers derived from the
model's predicted-risk distribution. They are not regulatory credit grades
and should not be interpreted as fixed universal PD thresholds.

### 7. External validation

External BNPL research and regulatory evidence are used for directional
validation and contextual interpretation. Differences in populations,
products and definitions prevent direct absolute-rate benchmarking.

### Responsible-use principle

The outputs should support human decision-making through differentiated
monitoring, exposure management, collections prioritisation, risk-adjusted
growth and stress preparedness.

They should not be used as automatic approval or rejection decisions without
additional validation, governance, policy controls and real-world monitoring.

**Core principle:**

**Use the analytics to improve risk decisions, not to replace risk governance.**

## 4.15 Final Executive Summary

This section consolidates the major analytical findings into a single executive
view of BNPL portfolio risk.

The final assessment integrates:
- supervised default-risk modelling
- portfolio risk concentration
- customer behavioural segmentation
- early-warning indicators
- expected-loss estimation
- macro stress testing
- strategic management recommendations

### Executive narrative

The analysis demonstrates that portfolio risk should not be managed through a
single average default measure. Risk is concentrated unevenly across the
portfolio, making exposure-weighted risk metrics and risk-tiered management
more informative for executive decision-making.

The highest-risk band represents a disproportionate share of model-derived
risk relative to its exposure share. This supports tighter exposure controls
and collections prioritisation for the highest-risk population, while lower-risk
portions of the portfolio provide the more appropriate basis for controlled
risk-adjusted growth.

Customer segmentation provides an additional behavioural lens, while temporal
risk indicators demonstrate the value of monitoring repayment deterioration
and delinquency signals.

Expected-loss analysis translates model-derived PD into economic exposure
through PD × EAD × scenario LGD. Stress testing then demonstrates how portfolio
losses could increase under adverse assumptions.

### Final decision principle

**Identify risk → quantify exposure → prioritise intervention → protect capital
→ enable risk-adjusted growth**

The results should be interpreted as decision-support evidence from a synthetic
BNPL portfolio, subject to the limitations documented in Section 4.14.

In [0]:
# ============================================================
# 4.15 Final Executive Summary
# ============================================================

from pyspark.sql import functions as F


# ------------------------------------------------------------
# 1. Core portfolio KPIs
# ------------------------------------------------------------

portfolio_kpis = (
    portfolio_risk
    .agg(
        F.count("*").alias("transaction_count"),
        F.countDistinct("customer_id").alias("customer_count"),
        F.sum("ead_ngn").alias("total_ead_ngn"),
        F.avg("portfolio_pd").alias("transaction_weighted_pd"),
        F.sum(
            F.col("portfolio_pd") * F.col("ead_ngn")
        ).alias("pd_weighted_exposure_ngn")
    )
    .first()
)

transaction_count = int(portfolio_kpis["transaction_count"])
customer_count = int(portfolio_kpis["customer_count"])
total_ead = float(portfolio_kpis["total_ead_ngn"])
transaction_weighted_pd = float(
    portfolio_kpis["transaction_weighted_pd"]
)
pd_weighted_exposure = float(
    portfolio_kpis["pd_weighted_exposure_ngn"]
)

exposure_weighted_pd = (
    pd_weighted_exposure / total_ead
    if total_ead > 0
    else None
)


# ------------------------------------------------------------
# 2. Baseline expected loss
# ------------------------------------------------------------

baseline_row = (
    expected_loss
    .filter(F.col("lgd_scenario") == "Baseline")
    .first()
)

if baseline_row is None:
    raise ValueError("Baseline expected-loss scenario was not found.")

baseline_lgd = float(baseline_row["lgd"])
baseline_el = float(baseline_row["expected_loss_ngn"])

el_rate_of_ead = (
    baseline_el / total_ead
    if total_ead > 0
    else None
)


# ------------------------------------------------------------
# 3. Very High risk concentration
# ------------------------------------------------------------

vh_row = (
    risk_band_concentration
    .filter(F.col("risk_band") == "Very High")
    .first()
)

if vh_row is None:
    raise ValueError("Very High risk band was not found.")

vh_ead = float(vh_row["ead_ngn"])
vh_ead_share = float(vh_row["ead_share_pct"])
vh_average_pd = float(vh_row["average_pd"])


# ------------------------------------------------------------
# 4. Very High expected-loss concentration
# ------------------------------------------------------------

vh_el = (
    portfolio_risk
    .filter(F.col("risk_band") == "Very High")
    .agg(
        F.sum(
            F.col("portfolio_pd")
            * F.col("ead_ngn")
            * F.lit(baseline_lgd)
        ).alias("very_high_el")
    )
    .first()["very_high_el"]
)

vh_el = float(vh_el)

vh_el_share = (
    vh_el / baseline_el
    if baseline_el > 0
    else None
)


# ------------------------------------------------------------
# 5. Risk concentration ratio
# ------------------------------------------------------------

vh_pd_weighted_exposure = (
    portfolio_risk
    .filter(F.col("risk_band") == "Very High")
    .agg(
        F.sum(
            F.col("portfolio_pd") * F.col("ead_ngn")
        ).alias("vh_pd_ead")
    )
    .first()["vh_pd_ead"]
)

vh_pd_weighted_exposure = float(vh_pd_weighted_exposure)

vh_pd_weighted_exposure_share = (
    vh_pd_weighted_exposure / pd_weighted_exposure
    if pd_weighted_exposure > 0
    else None
)

risk_concentration_ratio = (
    vh_pd_weighted_exposure_share
    / (vh_ead_share / 100.0)
    if vh_ead_share > 0
    else None
)


# ------------------------------------------------------------
# 6. Stress-test indicators
# ------------------------------------------------------------

moderate_row = (
    stress_summary
    .filter(F.lower(F.col("scenario")).contains("moderate"))
    .first()
)

severe_row = (
    stress_summary
    .filter(F.lower(F.col("scenario")).contains("severe"))
    .first()
)

if moderate_row is None:
    raise ValueError("Moderate macro stress scenario was not found.")

if severe_row is None:
    raise ValueError("Severe macro stress scenario was not found.")

moderate_el = float(
    moderate_row["stressed_expected_loss_ngn"]
)

severe_el = float(
    severe_row["stressed_expected_loss_ngn"]
)

moderate_el_multiple = (
    moderate_el / baseline_el
    if baseline_el > 0
    else None
)

severe_el_multiple = (
    severe_el / baseline_el
    if baseline_el > 0
    else None
)


# ------------------------------------------------------------
# 7. Final executive KPI scorecard
# ------------------------------------------------------------

final_kpis = [
    (
        "Total Portfolio EAD",
        total_ead,
        "NGN",
        "Total principal-based exposure represented in the portfolio."
    ),
    (
        "Exposure-Weighted PD",
        exposure_weighted_pd * 100,
        "%",
        "Economically weighted model-derived default risk."
    ),
    (
        "Baseline Expected Loss",
        baseline_el,
        "NGN",
        "Scenario-based expected loss using baseline LGD."
    ),
    (
        "Expected Loss / EAD",
        el_rate_of_ead * 100,
        "%",
        "Baseline expected loss relative to portfolio exposure."
    ),
    (
        "Very High Risk EAD Share",
        vh_ead_share,
        "%",
        "Share of total exposure concentrated in the Very High band."
    ),
    (
        "Very High Expected Loss Share",
        vh_el_share * 100,
        "%",
        "Share of baseline expected loss contributed by the Very High band."
    ),
    (
        "Risk Concentration Ratio",
        risk_concentration_ratio,
        "x",
        "Very High PD-weighted exposure share relative to its EAD share."
    ),
    (
        "Moderate Stress Expected Loss",
        moderate_el,
        "NGN",
        "Expected loss under the moderate macro stress scenario."
    ),
    (
        "Severe Stress Expected Loss",
        severe_el,
        "NGN",
        "Expected loss under the severe macro stress scenario."
    ),
    (
        "Severe / Baseline Expected Loss",
        severe_el_multiple,
        "x",
        "Increase in expected loss under severe stress relative to baseline."
    )
]

final_kpi_df = spark.createDataFrame(
    final_kpis,
    [
        "executive_kpi",
        "value",
        "unit",
        "executive_meaning"
    ]
)

display(final_kpi_df)


# ------------------------------------------------------------
# 8. Final management conclusions
# ------------------------------------------------------------

print("FINAL EXECUTIVE SUMMARY")
print("=" * 75)

print("\nPORTFOLIO POSITION")
print("-" * 75)

print(
    f"Transactions: {transaction_count:,}"
)

print(
    f"Customers: {customer_count:,}"
)

print(
    f"Total EAD: ₦{total_ead:,.2f}"
)

print(
    f"Exposure-weighted PD: "
    f"{exposure_weighted_pd * 100:.2f}%"
)

print(
    f"Baseline expected loss: "
    f"₦{baseline_el:,.2f}"
)

print(
    f"Expected loss / EAD: "
    f"{el_rate_of_ead * 100:.2f}%"
)


print("\nRISK CONCENTRATION")
print("-" * 75)

print(
    f"Very High risk EAD share: "
    f"{vh_ead_share:.2f}%"
)

print(
    f"Very High average PD: "
    f"{vh_average_pd * 100:.2f}%"
)

print(
    f"Very High expected-loss share: "
    f"{vh_el_share * 100:.2f}%"
)

print(
    f"Risk concentration ratio: "
    f"{risk_concentration_ratio:.2f}x"
)


print("\nSTRESS RESILIENCE")
print("-" * 75)

print(
    f"Moderate stress expected loss: "
    f"₦{moderate_el:,.2f}"
)

print(
    f"Moderate / baseline EL: "
    f"{moderate_el_multiple:.2f}x"
)

print(
    f"Severe stress expected loss: "
    f"₦{severe_el:,.2f}"
)

print(
    f"Severe / baseline EL: "
    f"{severe_el_multiple:.2f}x"
)


print("\nEXECUTIVE CONCLUSIONS")
print("-" * 75)

print(
    "1. Portfolio risk is concentrated unevenly, so exposure-weighted "
    "metrics are more informative than a simple average default measure."
)

print(
    "2. The Very High risk band warrants tighter incremental exposure "
    "controls and priority collections intervention."
)

print(
    "3. Lower-risk bands provide the more appropriate basis for "
    "controlled risk-adjusted portfolio growth."
)

print(
    "4. Behavioural segmentation and deterioration indicators provide "
    "additional signals for differentiated monitoring."
)

print(
    "5. Expected-loss analysis converts model-derived risk into an "
    "economic portfolio view through PD × EAD × LGD."
)

print(
    "6. Stress testing demonstrates that adverse assumptions can produce "
    "material increases in expected loss, supporting contingency planning."
)

print(
    "\nFINAL MANAGEMENT MESSAGE"
)

print(
    "The recommended strategy is to protect capital in concentrated "
    "high-risk exposures while enabling controlled growth in lower-risk "
    "portfolio segments, supported by differentiated monitoring, "
    "collections prioritisation and stress preparedness."
)

print(
    "\nResponsible-use boundary:"
)

print(
    "All results are decision-support outputs from a synthetic BNPL "
    "portfolio and should not be interpreted as empirical Nigerian market "
    "estimates, regulatory credit grades, causal macro forecasts or "
    "automatic lending decisions."
)

executive_kpi,value,unit,executive_meaning
Total Portfolio EAD,3.3317913042690517E10,NGN,Total principal-based exposure represented in the portfolio.
Exposure-Weighted PD,9.87359237746511,%,Economically weighted model-derived default risk.
Baseline Expected Loss,1.3158699690054088E9,NGN,Scenario-based expected loss using baseline LGD.
Expected Loss / EAD,3.949436950986017,%,Baseline expected loss relative to portfolio exposure.
Very High Risk EAD Share,25.172716335120963,%,Share of total exposure concentrated in the Very High band.
Very High Expected Loss Share,62.13179488447109,%,Share of baseline expected loss contributed by the Very High band.
Risk Concentration Ratio,2.4682197208008123,x,Very High PD-weighted exposure share relative to its EAD share.
Moderate Stress Expected Loss,2.2616515092280593E9,NGN,Expected loss under the moderate macro stress scenario.
Severe Stress Expected Loss,3.4541586686392164E9,NGN,Expected loss under the severe macro stress scenario.
Severe / Baseline Expected Loss,2.625000000000014,x,Increase in expected loss under severe stress relative to baseline.


FINAL EXECUTIVE SUMMARY

PORTFOLIO POSITION
---------------------------------------------------------------------------
Transactions: 666,246
Customers: 421,457
Total EAD: ₦33,317,913,042.69
Exposure-weighted PD: 9.87%
Baseline expected loss: ₦1,315,869,969.01
Expected loss / EAD: 3.95%

RISK CONCENTRATION
---------------------------------------------------------------------------
Very High risk EAD share: 25.17%
Very High average PD: 21.12%
Very High expected-loss share: 62.13%
Risk concentration ratio: 2.47x

STRESS RESILIENCE
---------------------------------------------------------------------------
Moderate stress expected loss: ₦2,261,651,509.23
Moderate / baseline EL: 1.72x
Severe stress expected loss: ₦3,454,158,668.64
Severe / baseline EL: 2.63x

EXECUTIVE CONCLUSIONS
---------------------------------------------------------------------------
1. Portfolio risk is concentrated unevenly, so exposure-weighted metrics are more informative than a simple average default measure.
2. 